In [1]:
import os, pathlib
print(os.getcwd())
print('\n'.join(str(p) for p in pathlib.Path('/workspace').iterdir()))

/workspace
/workspace/truth_tables.md
/workspace/settings.md
/workspace/memory
/workspace/lost+found
/workspace/results
/workspace/analyze.py
/workspace/quality
/workspace/compare
/workspace/.config
/workspace/plot.py
/workspace/.skills
/workspace/bofn_cu_best_vs_spend_k1_partial_vs_full.png
/workspace/.kernel_llm_logs_1.jsonl
/workspace/bofn_alloy.py
/workspace/bofn_cu_best_vs_spend_k1_partial_vs_full_current.png
/workspace/.prompts
/workspace/compare_tables.md
/workspace/policies.py
/workspace/README.md
/workspace/truth_tables.json
/workspace/tables.md


In [2]:
from pathlib import Path
import json
for folder in ['results','quality','compare','.prompts']:
 print(folder,[(str(p),p.stat().st_size) for p in Path(folder).rglob('*') if p.is_file()][:25])
print('README\n',Path('README.md').read_text())
print('TABLES\n',Path('tables.md').read_text())
f=next(Path('results/k1e1').glob('pkgfn_s*.json')); d=json.loads(f.read_text());print(f, d.keys());print(str(d)[:8000])

results [('results/k0e0/eifn_s0.json', 17240), ('results/k0e0/eifn_reuse_s0.json', 20410), ('results/k1e1/ei_s4.json', 16974), ('results/k1e1/pkgfn_s5.json', 37703), ('results/k1e1/eifn_s10.json', 17289), ('results/k1e1/tiered_ni_ref_s8.json', 33262), ('results/k1e1/ei_s18.json', 17072), ('results/k1e1/tiered_ni_s2.json', 37939), ('results/k1e1/ei_s6.json', 17205), ('results/k1e1/tiered_s15.json', 35776), ('results/k1e1/tiered_ni_ref_s16.json', 36836), ('results/k1e1/random_s0.json', 17569), ('results/k1e1/eifn_reuse_s14.json', 19189), ('results/k1e1/eifn_reuse_s7.json', 19334), ('results/k1e1/eifn_reuse_s2.json', 20268), ('results/k1e1/random_s15.json', 17763), ('results/k1e1/eifn_s18.json', 17304), ('results/k1e1/eifn_s8.json', 17320), ('results/k1e1/tiered_ni_ref_s14.json', 22043), ('results/k1e1/eifn_reuse_s17.json', 20099), ('results/k1e1/random_s7.json', 17700), ('results/k1e1/ei_s16.json', 16771), ('results/k1e1/pkgfn_s2.json', 38450), ('results/k1e1/tiered_ni_s19.json', 20412),

In [3]:
import numpy as np, scipy, scipy.stats as st, collections, pandas as pd
out=Path('/workspace/outputs');out.mkdir(exist_ok=True)
truth=json.loads(Path('truth_tables.json').read_text());print(str(truth)[:4500])
print('schema keys',d.keys())
print('metadata',{k:v for k,v in d.items() if k not in ['rows','builds','lots']})
for k in ['rows','counts','builds','lots']:
 v=d.get(k); print(k,type(v),len(v) if v is not None else None,str(v)[:1800])
print('inventory',collections.Counter((p.parent.name,p.stem.rsplit('_s',1)[0]) for p in Path('results').rglob('*.json')))
print('compare files',[str(p) for p in Path('compare').glob('*')])

[{'kappa': 1.0, 'eta': 1.0, 'optimum': 0.492187251856002, 'x_opt': [0.434, 0.585, 0.5, 0.0, 0.643, 0.582, 0.343], 'nodes_opt': {'s': 0.728, 'f': 0.737, 'g': 0.408, 'h': 0.125, 'q': 0.81, 'rho_u': 0.794, 'rho_m': 0.932, 'beta': 0.153, 'U': 0.759, 'w': 0.884, 'm': 0.953}, 'burden_opt': 0.6093136765285487, 'optimum_fails_calphad_screen': True, 'F_SAFE': 0.34, 'w_gate_top_quartile_of_dense': 0.6988394908994753, 'optimum_w_percentile_among_dense': 1.0, 'optimum_below_top_quartile': False, 'best_passing_calphad_screen': {'J': 0.35538767462610227, 'pct': 0.7220578616897562, 'x': [0.285, 0.352, 0.515, 0.002, 0.48, 0.707, 0.576], 'nodes': {'s': 0.52, 'f': 0.34, 'g': 0.366, 'h': 0.242, 'q': 0.959, 'rho_u': 0.311, 'rho_m': 0.965, 'beta': 0.11, 'U': 0.628, 'w': 0.773, 'm': 0.905}}, 'best_passing_both_screens': {'J': 0.3547451492268922, 'pct': 0.7207524126014526, 'x': [0.292, 0.327, 0.495, 0.0, 0.49, 0.722, 0.588], 'nodes': {'s': 0.504, 'f': 0.339, 'g': 0.348, 'h': 0.241, 'q': 0.959, 'rho_u': 0.273

In [4]:
# Independent implementation, deliberately BEFORE reading or importing analyze.py.
# Plan: reconstruct right-continuous curves; summarize and pair by explicit seed IDs;
# bootstrap crossing ratios and fixed targets; then compare supplied code/tables;
# audit actions, budgets, validation completeness, sensitivities, and both PNGs.
OPT={(int(t['kappa']),int(t['eta'])):t['optimum'] for t in truth}
GRID=np.arange(0,100001,250,dtype=float)
runs=collections.defaultdict(dict)
for root,override in [('results',None),('quality/results-light','pkgfn_light')]:
 for p in sorted(Path(root).rglob('*.json')):
  r=json.loads(p.read_text());r['_path']=str(p)
  key=(int(r['kappa']),int(r['eta']),override or r['kind'])
  assert r['seed'] not in runs[key]
  runs[key][r['seed']]=r

def independent_curve(run,cell,metric='best_validated',grid=GRID):
 rows=run['rows']; spends=np.array([x['spent'] for x in rows],float)
 assert np.all(np.diff(spends)>=0)
 vals=np.array([x[metric] for x in rows],float)
 assert np.all(np.isfinite(vals))
 vals=np.maximum.accumulate(vals)
 # Shared initial observations are represented by the initial row (checked below).
 idx=np.searchsorted(spends,grid,side='right')-1
 return vals[np.maximum(idx,0)]/OPT[cell]*100

def crossing(curve,target):
 hits=np.flatnonzero(curve>=target)
 return GRID[hits[0]] if len(hits) else np.inf

curves={key:np.array([independent_curve(r,key[:2]) for s,r in sorted(v.items())]) for key,v in runs.items()}
seedids={key:np.array(sorted(v)) for key,v in runs.items()}
rows=[]
for key,a in curves.items():
 row={'cell':str(key[:2]),'policy':key[2],'n':len(a),'start':a[:,0].mean()}
 for b in [25000,50000,100000]:
  v=a[:,int(b/250)];row[f'mean{b}']=v.mean();row[f'se{b}']=st.sem(v) if len(v)>1 else np.nan
 for t in [60,70,80]:
  costs=np.array([crossing(c,t) for c in a]);row[f'median{t}']=np.median(costs);row[f'hit{t}']=np.isfinite(costs).mean()
 rows.append(row)
summary=pd.DataFrame(rows)
summary.to_csv(out/'independent_summary.csv',index=False)
print(summary.to_string(index=False,float_format=lambda x:f'{x:.5f}'))
print('Initial rows',collections.Counter((r['rows'][0]['spent'],r['rows'][0].get('action')) for v in runs.values() for r in v.values()))
# Save the independent implementation before supplied analysis is opened.
import IPython
Path(out/'independent_core.py').write_text(IPython.get_ipython().history_manager.input_hist_raw[-1].split('# Save the independent')[0])


  cell        policy  n     start  mean25000  se25000  mean50000  se50000  mean100000  se100000     median60   hit60    median70   hit70    median80   hit80
(0, 0)    eifn_reuse  1  11.83111   11.83111      NaN   67.67248      NaN    95.19460       NaN  46500.00000 1.00000 53500.00000 1.00000 58250.00000 1.00000
(0, 0)          eifn  1  11.83111   11.83111      NaN   19.48662      NaN    92.25496       NaN  52500.00000 1.00000 71500.00000 1.00000 76250.00000 1.00000
(1, 1)            ei 20  -9.60955    5.68040  3.41324    7.18040  3.29873    14.42051   6.25620          inf 0.15000         inf 0.05000         inf 0.05000
(1, 1)    eifn_reuse 20  -9.60955   14.35254  4.70224   32.27423  7.80548    72.03264   7.04636  54375.00000 0.85000 64625.00000 0.85000 80625.00000 0.65000
(1, 1)          eifn 20  -9.60955   14.97670  5.74505   38.77389  9.11598    73.21174   7.32195  66750.00000 0.85000 73875.00000 0.75000 88125.00000 0.70000
(1, 1)         pkgfn  7 -13.83079    3.34113  5.43279   24

2210

In [5]:
POL=['pkgfn','pkgfn_light','eifn','eifn_reuse','tiered','tiered_ni','tiered_ni_ref','ei','random']
BUD=[25000,50000,100000]
paired_rows=[]
for pol in POL:
 if pol=='pkgfn':continue
 k1=(1,1,'pkgfn'); k2=(1,1,pol)
 common=np.intersect1d(seedids[k1],seedids[k2])
 a=np.array([curves[k1][np.where(seedids[k1]==s)[0][0]] for s in common])
 b=np.array([curves[k2][np.where(seedids[k2]==s)[0][0]] for s in common])
 for bud in BUD:
  diff=(a-b)[:,int(bud/250)]
  nonzero=diff[np.abs(diff)>1e-10]
  paired_rows.append({'against':pol,'budget':bud,'n':len(diff),'difference':diff.mean(),'wins':sum(diff>1e-10),'ties':sum(abs(diff)<=1e-10),'losses':sum(diff< -1e-10),'p_dropzero':st.wilcoxon(nonzero).pvalue if len(nonzero) else 1.,'p_scipy_default':st.wilcoxon(diff).pvalue if len(nonzero) else 1.})
paired_results=pd.DataFrame(paired_rows);print(paired_results.to_string(index=False));paired_results.to_csv(out/'independent_paired.csv',index=False)

# Bootstrap: seed units; each replicate resets target to resampled baseline final mean.
# Explicit inf for non-crossing. Independently resample all observed seeds OR resample
# common seed IDs jointly; the latter necessarily changes the sample for 7-v-20 runs.
def first_cross_batch(a,t):
 hit=a>=np.asarray(t)[:,None]
 return np.where(hit.any(1),GRID[hit.argmax(1)],np.inf)
def boot_ratio(method,baseline,matched=False,nboot=20000,rngseed=20261027):
 k1=(1,1,method);k2=(1,1,baseline);a=curves[k1];b=curves[k2]
 if matched:
  common=np.intersect1d(seedids[k1],seedids[k2]);a=a[np.searchsorted(seedids[k1],common)];b=b[np.searchsorted(seedids[k2],common)]
 rng=np.random.default_rng(rngseed);ratios=[];zero=[]
 for start in range(0,nboot,1000):
  n=min(1000,nboot-start)
  ia=rng.integers(0,len(a),(n,len(a)));ib=ia if matched else rng.integers(0,len(b),(n,len(b)))
  ma=a[ia].mean(1); mb=b[ib].mean(1);tar=mb[:,-1]
  ca=first_cross_batch(ma,tar);cb=first_cross_batch(mb,tar)
  with np.errstate(divide='ignore',invalid='ignore'):r=cb/ca
  ratios.extend(r);zero.extend(ca==0)
 return np.array(ratios),np.mean(zero)
ratio_rows=[];boot_store={}
for base in ['tiered','tiered_ni_ref']:
 b=curves[1,1,base].mean(0);target=b[-1];bc=crossing(b,target)
 for pol in POL:
  if pol==base: continue
  a=curves[1,1,pol].mean(0);ac=crossing(a,target)
  row={'baseline':base,'policy':pol,'target':target,'baseline_cost':bc,'method_cost':ac,'ratio':bc/ac}
  for match in [False,True]:
   rr,z=boot_ratio(pol,base,match);boot_store[base,pol,match]=rr
   valid=rr[~np.isnan(rr)]
   # inverted empirical CDF avoids interpolation between finite and infinite values.
   qs=np.quantile(valid,[.05,.95],method='inverted_cdf')
   pref='matched' if match else 'independent'
   row.update({pref+'_lo':qs[0],pref+'_hi':qs[1],pref+'_undefined':np.isnan(rr).mean(),pref+'_zero_cost':z,pref+'_Pge3':np.mean(rr>=3)})
  ratio_rows.append(row)
ratio_results=pd.DataFrame(ratio_rows);ratio_results.to_csv(out/'independent_speedups.csv',index=False)
print(ratio_results.to_string(index=False,float_format=lambda x:f'{x:.5f}'))


      against  budget  n  difference  wins  ties  losses  p_dropzero  p_scipy_default
  pkgfn_light   25000  7  -11.322800     3     0       4    0.578125         0.578125
  pkgfn_light   50000  7   -2.099425     3     0       4    0.937500         0.937500
  pkgfn_light  100000  7   -5.336754     3     0       4    0.687500         0.687500
         eifn   25000  7  -20.507426     0     0       7    0.015625         0.015625
         eifn   50000  7  -20.773717     1     0       6    0.218750         0.218750
         eifn  100000  7  -28.510794     1     0       6    0.046875         0.046875
   eifn_reuse   25000  7  -14.006848     1     0       6    0.109375         0.109375
   eifn_reuse   50000  7  -14.940369     2     0       5    0.375000         0.375000
   eifn_reuse  100000  7   -3.053437     3     0       4    0.812500         0.812500
       tiered   25000  7   16.649669     6     1       0    0.031250         0.031250
       tiered   50000  7   36.963451     7     0      

     baseline        policy   target  baseline_cost  method_cost    ratio  independent_lo  independent_hi  independent_undefined  independent_zero_cost  independent_Pge3  matched_lo  matched_hi  matched_undefined  matched_zero_cost  matched_Pge3
       tiered         pkgfn -7.69082    99250.00000   5750.00000 17.26087         2.51370             inf                0.00040                0.24905           0.93875    11.52632    20.78947            0.00200            0.00200       0.99800
       tiered   pkgfn_light -7.69082    99250.00000   5750.00000 17.26087         5.75362             inf                0.00085                0.39830           0.99140    12.03030    21.94444            0.00255            0.00255       0.99745
       tiered          eifn -7.69082    99250.00000   5000.00000 19.85000        18.35000             inf                0.00085                0.39830           0.99575    18.35000    19.85000            0.00255            0.00255       0.99745
       tiered   

In [6]:
import ast, difflib
source=Path('analyze.py').read_text();tree=ast.parse(source)
# Execute supplied pure-NumPy analysis with only heavyweight import statements removed.
# optimum reads the uploaded constants, so no torch or optimization is invoked.
body=[]
for node in tree.body:
 if isinstance(node,ast.Import) and any(x.name=='torch' for x in node.names):continue
 if isinstance(node,ast.ImportFrom) and node.module=='bofn_alloy':continue
 if isinstance(node,ast.If) and isinstance(node.test,ast.Compare) and '__name__' in ast.unparse(node.test):continue
 body.append(node)
COST={'C':10.,'A':400.,'U':500.,'M':1500.,'S':100.,'T':200.,'K':50.,'I':400.}
ana={'__file__':str(Path('analyze.py').resolve()),'COSTS':COST,'M_REF':d['m_ref'],'D':7}
exec(compile(ast.Module(body=body,type_ignores=[]),'analyze_numpy_only','exec'),ana)
regen=ana['report']();(out/'supplied_tables_regenerated.md').write_text(regen)
diff=list(difflib.unified_diff(Path('tables.md').read_text().splitlines(),regen.splitlines(),fromfile='uploaded tables.md',tofile='regenerated'))
print('DIFF count',len(diff));print('\n'.join(diff))
# Compare every independent best_validated curve to supplied curves, seed-by-seed.
maxerr=0
for key,a in curves.items():
 supplied=ana['curves'](ana['load'](key[:2])[key[2]],key[:2])*100
 assert supplied.shape==a.shape
 maxerr=max(maxerr,float(np.nanmax(abs(supplied-a))))
print('Max curve difference (percentage points)',maxerr)
print('SciPy',scipy.__version__)
print('SE correction factors',[(n,np.sqrt(n/(n-1))) for n in [7,10,20]])
# independent mean table with formatting matching supplied conventions except sample SE
for pol in POL:
 row=summary.query("cell=='(1, 1)' and policy==@pol").iloc[0]
 print(pol,'; '.join(f'{row[f"mean{b}"]:.0f}% ± {row[f"se{b}"]:.0f}' for b in BUD))


DIFF count 0

Max curve difference (percentage points) 0
SciPy 1.17.1
SE correction factors [(7, np.float64(1.0801234497346435)), (10, np.float64(1.0540925533894598)), (20, np.float64(1.025978352085154))]
pkgfn 3% ± 5; 25% ± 11; 59% ± 12
pkgfn_light 12% ± 7; 27% ± 7; 59% ± 5
eifn 15% ± 6; 39% ± 9; 73% ± 7
eifn_reuse 14% ± 5; 32% ± 8; 72% ± 7
tiered -9% ± 6; -9% ± 5; -8% ± 5
tiered_ni -7% ± 5; 1% ± 6; 10% ± 6
tiered_ni_ref 8% ± 4; 16% ± 4; 24% ± 5
ei 6% ± 3; 7% ± 3; 14% ± 6
random -1% ± 5; 7% ± 5; 22% ± 5


In [7]:
print('counts log schemas by policy')
for pol in POL:
 r=runs[1,1,pol][min(runs[1,1,pol])]
 print('\n',pol,'keys',list(r),'rows',len(r['rows']),'actions',dict(collections.Counter(x['action'] for x in r['rows'])))
 print('build keys',r.get('builds',[{}])[0].keys(),'lastbuild',str(r.get('builds',[{}])[-1])[:500])
 print('lot keys',r.get('lots',[{}])[0].keys(),'events',str(r.get('events',[]))[:350])
print('policy function lines')
ptext=Path('policies.py').read_text().splitlines()
for i,l in enumerate(ptext,1):
 if any(s in l for s in ['def ','counts','best_validated','fallback','rows.append','break','sessions_left','def purchase']):
  print(i,l)


counts log schemas by policy

 pkgfn keys ['kind', 'kappa', 'eta', 'seed', 'budget', 'm_ref', 'rows', 'counts', 'settings', 'gp_fits', 'builds', 'lots', 'w_gate_final', 'events', 'wall_s', '_path'] rows 87 actions {'init': 1, 'M': 44, 'C': 8, 'A': 7, 'S+K+I': 12, 'U': 7, 'T': 3, 'S+T': 2, 'K': 1, 'K+I': 1, 'S': 1}
build keys dict_keys(['c', 'a', 'pv', 'q', 'rho_u_in', 'rho_m', 'beta', 'U', 'w', 'm', 'J']) lastbuild {'c': [0.7612, 0.6999, 0.448, 0.4431], 'a': 0.5203814760846887, 'pv': [0.5345384115353227, 0.11808257270604372], 'q': 0.3170765427959693, 'rho_u_in': 0.6346560830723116, 'rho_m': 0.49914769321996566, 'beta': None, 'U': None, 'w': None, 'm': None, 'J': None}
lot keys dict_keys(['c', 'a', 'q', 'utah', 'sessions_left']) events [{'it': 7, 'build': 8, 'w': 0.9227099291860619, 'gate': 0.8852095677617525, 'screen_pass': True, 'how': 'pass-bundle', 'm': 0.7814019885946341, 'immersion_pass': False}, {'it': 9, 'build': 9, 'w': 0.7093165464728123, 'gate': 0.8795138894571175, 'screen_pa

In [8]:
# Execute edge cases on the supplied functions.
edge=[{'spent':500.,'best_validated':.2},{'spent':1000.,'best_validated':.1},
      {'spent':1000.,'best_validated':.3},{'spent':100000.,'best_validated':.4},
      {'spent':100010.,'best_validated':.9}]
ec=ana['curve'](edge)
print('Edge curve at 0,250,500,750,1000,99750,100000:',ec[[0,1,2,3,4,399,400]])
print('Edge cost_to .2,.3,.4,.5:',[ana['cost_to'](ec[None,:],t).tolist() for t in [.2,.3,.4,.5]])
A=[{'seed':7},{'seed':0},{'seed':2}];BB=[{'seed':2},{'seed':3},{'seed':7}]
ip,ib=ana['matched_idx'](A,BB);print('shuffled seed match',ip,ib,[(A[i]['seed'],BB[j]['seed']) for i,j in zip(ip,ib)])
print('actual duplicate seeds',len(seedids),all(len(np.unique(ss))==len(ss) for ss in seedids.values()))
# Independent exact replication of the stated estimator including the denominator floor.
def iratio(a,b):
 target=b[:,-1].mean();bc=crossing(b.mean(0),target);ac=crossing(a.mean(0),target)
 return bc/max(ac,250.)
def exact_independent_boot(pol,base,n=2000,seed=0):
 a=curves[1,1,pol]/100;b=curves[1,1,base]/100
 # tiny epsilon matches supplied first_cost and only addresses floating point equality
 def cr(a,t):
  ii=np.flatnonzero(a>=t-1e-12);return GRID[ii[0]] if len(ii) else np.inf
 def rat(a,b):
  t=b[:,-1].mean();return cr(b.mean(0),t)/max(cr(a.mean(0),t),250.)
 rng=np.random.default_rng(seed)
 vals=np.array([rat(a[rng.integers(0,len(a),len(a))],b[rng.integers(0,len(b),len(b))]) for _ in range(n)])
 com=np.intersect1d(seedids[1,1,pol],seedids[1,1,base]);ai=np.searchsorted(seedids[1,1,pol],com);bi=np.searchsorted(seedids[1,1,base],com)
 valsm=[]
 for _ in range(n):
  j=rng.integers(0,len(com),len(com));valsm.append(rat(a[ai[j]],b[bi[j]]))
 return {'est':rat(a,b),'lo':np.percentile(vals,5),'hi':np.percentile(vals,95),'p3':np.mean(vals>=3),'mlo':np.percentile(valsm,5),'mhi':np.percentile(valsm,95)}, rat(a[ai],b[bi]),b[bi,-1].mean()*100
exactboot=[]
for base in ['tiered','tiered_ni_ref']:
 for pol in POL:
  if pol==base:continue
  vals,matchedpoint,matchedtarget=exact_independent_boot(pol,base)
  expected=ana['speedup'](curves[1,1,pol]/100,curves[1,1,base]/100,matched=ana['matched_idx'](ana['load']((1,1))[pol],ana['load']((1,1))[base]))
  assert all(np.isclose(vals[k],expected[k]) for k in vals)
  exactboot.append({'base':base,'policy':pol,**vals,'matched_point':matchedpoint,'matched_target':matchedtarget})
exactboot=pd.DataFrame(exactboot);print(exactboot.to_string(index=False));exactboot.to_csv(out/'replicated_bootstrap.csv',index=False)


Edge curve at 0,250,500,750,1000,99750,100000: [nan nan 0.2 0.2 0.3 0.3 0.4]
Edge cost_to .2,.3,.4,.5: [[500.0], [1000.0], [100000.0], [inf]]
shuffled seed match [2 0] [0 2] [(2, 2), (7, 7)]
actual duplicate seeds 12 True


         base        policy       est        lo         hi     p3       mlo       mhi  matched_point  matched_target
       tiered         pkgfn 17.260870  2.719178 397.000000 0.9485 11.526316 20.789474      20.789474      -10.434198
       tiered   pkgfn_light 17.260870  5.753623 397.000000 0.9885 11.969697 21.944444      17.260870       -7.690822
       tiered          eifn 19.850000 18.350000 397.000000 0.9925 18.350000 19.850000      19.850000       -7.690822
       tiered    eifn_reuse 19.850000 10.950000 397.000000 0.9955 18.350000 43.888889      19.850000       -7.690822
       tiered     tiered_ni  4.268817  1.150725 397.000000 0.5535  2.682863  7.218182       4.268817       -7.690822
       tiered tiered_ni_ref 11.027778  5.025316 397.000000 0.9870  7.218182 14.178571      11.027778       -7.690822
       tiered            ei 19.850000  5.155844 397.000000 0.9760 18.350000 19.850000      19.850000       -7.690822
       tiered        random 10.179487  2.308140 397.000000 0.869

In [9]:
target_rows=[]
for pol in POL:
 m=curves[1,1,pol].mean(0)
 target_rows.append({'policy':pol,**{f'cost{t}':crossing(m,t) for t in [0,24.233129,55,60,65]}})
targetdf=pd.DataFrame(target_rows);print('Fixed target first crossings',targetdf.to_string(index=False));targetdf.to_csv(out/'fixed_target_costs.csv',index=False)
# Holm adjustment, first the requested 8 x 3 family, then all 36 x 3 policy pairs.
def holm_adjust(p):
 p=np.asarray(p);order=np.argsort(p);r=np.maximum.accumulate(p[order]*(len(p)-np.arange(len(p))));outp=np.empty_like(p);outp[order]=np.minimum(1,r);return outp
paired_results['p_holm_24']=holm_adjust(paired_results.p_scipy_default)
print('Holm minimum',paired_results.p_holm_24.min())
print(paired_results[['against','budget','p_scipy_default','p_holm_24']].to_string(index=False))
allpairs=[]
import itertools
for p,q in itertools.combinations(POL,2):
 common=np.intersect1d(seedids[1,1,p],seedids[1,1,q]);a=curves[1,1,p][np.searchsorted(seedids[1,1,p],common)];b=curves[1,1,q][np.searchsorted(seedids[1,1,q],common)]
 for bud in BUD:
  dv=(a-b)[:,int(bud/250)];pv=st.wilcoxon(dv).pvalue if np.any(dv) else 1.
  allpairs.append({'p':p,'q':q,'budget':bud,'n':len(dv),'diff':dv.mean(),'wins':sum(dv>0),'ties':sum(dv==0),'losses':sum(dv<0),'pval':pv})
allpairs=pd.DataFrame(allpairs);allpairs['p_holm108']=holm_adjust(allpairs.pval)
print('Key BOFN comparisons',allpairs.query("p=='eifn' and budget==100000").to_string(index=False))
allpairs.to_csv(out/'all_policy_pairs.csv',index=False);paired_results.to_csv(out/'independent_paired.csv',index=False)
# matched high/light comparison means and endpoints including raw seed values
com=seedids[1,1,'pkgfn']
for pol in ['pkgfn','pkgfn_light','eifn','eifn_reuse','tiered_ni_ref']:
 a=curves[1,1,pol][np.searchsorted(seedids[1,1,pol],com)]
 print('Matched',pol,com,'values',np.round(a[:,-1],5),'mean ± sample SE',[(round(a[:,b//250].mean(),5),round(st.sem(a[:,b//250]),5)) for b in BUD], 'median60',np.median([crossing(c,60) for c in a]))
print('End-of-budget pkgfn')
for s,r in runs[1,1,'pkgfn'].items():
 a=independent_curve(r,(1,1));print(s,[(r0['spent'],r0['action'],round(r0['best_validated']/OPT[1,1]*100,5)) for r0 in r['rows'][-5:]],'last250_jump',a[-1]-a[-2])


Fixed target first crossings        policy   cost0  cost24.233129   cost55  cost60  cost65
        pkgfn 12000.0        47250.0 100000.0     inf     inf
  pkgfn_light 12250.0        46750.0 100000.0     inf     inf
         eifn  5000.0        33500.0  76250.0 85750.0 85750.0
   eifn_reuse  5000.0        35250.0  65750.0 71000.0 81000.0
       tiered     inf            inf      inf     inf     inf
    tiered_ni 47250.0            inf      inf     inf     inf
tiered_ni_ref 14750.0       100000.0      inf     inf     inf
           ei  9750.0            inf      inf     inf     inf
       random 28750.0            inf      inf     inf     inf
Holm minimum 0.375
      against  budget  p_scipy_default  p_holm_24
  pkgfn_light   25000         0.578125    1.00000
  pkgfn_light   50000         0.937500    1.00000
  pkgfn_light  100000         0.687500    1.00000
         eifn   25000         0.015625    0.37500
         eifn   50000         0.218750    1.00000
         eifn  100000         0.

In [10]:
# Action/inventory audit, excluding five fully characterized seed builds/lots.
audit=[];perseed=[];details=[]
for key,rs in runs.items():
 for seed,r in sorted(rs.items()):
  rr=r['rows'];build=r['builds'];new=build[5:];lots=r['lots'];counts=r['counts']
  cnt=len(rr)-1;fallback=sum(bool(x.get('fallback')) for x in rr[1:]);deltas=np.diff([x['spent'] for x in rr]);fb_spend=sum(delta for x,delta in zip(rr[1:],deltas) if x.get('fallback'))
  assert len(new)==counts['M']; assert len(lots)-5==counts['A']
  for stage,field in [('S','beta'),('T','U'),('K','w'),('I','m')]:
   assert sum(b[field] is not None for b in new)==counts[stage],(key,seed,stage)
  total=sum(counts[k]*v for k,v in COST.items());scaleups=(rr[-1]['spent']-total)/1600
  assert abs(scaleups-round(scaleups))<1e-8 and scaleups>=0
  assert rr[-1]['spent']<=r['budget']
  assert all(x['J'] is not None for x in build[:5])
  assert abs(max(b['J'] for b in build if b['J'] is not None)-rr[-1]['best_validated'])<1e-12
  # Exact equality in stored tuples (composition coordinates are rounded to four decimals).
  lotkey=lambda l:tuple(l['c'])+(l['a'],)
  bkey=lambda b:tuple(b['c'])+(b['a'],)+tuple(b['pv'])
  dup_lots=len(lots)-len(set(map(lotkey,lots)))
  dup_builds=len(build)-len(set(map(bkey,build)))
  ulist=[lotkey(l)+tuple(u[:2]) for l in lots for u in l['utah']]
  dup_u=len(ulist)-len(set(ulist))
  # Longest validated-objective plateau, until first improvement or end.
  lastgain=0.;best=rr[0]['best_validated'];maxplat=(0.,0.,0.);cheapn=0;cheapstart=0.;cheapspend=0.;maxcheap=(0,0.,0.,0.)
  for prev,x,delta in zip(rr,rr[1:],deltas):
   gain=x['best_validated']>best+1e-12
   if gain:
    if x['spent']-lastgain>maxplat[0]:maxplat=(x['spent']-lastgain,lastgain,x['spent'])
    lastgain=x['spent'];best=x['best_validated']
   if not gain and delta<=500:
    if cheapn==0:cheapstart=prev['spent'];cheapspend=0.
    cheapn+=1;cheapspend+=delta
    if cheapn>maxcheap[0]:maxcheap=(cheapn,cheapspend,cheapstart,x['spent'])
   else:cheapn=0;cheapspend=0.
  if rr[-1]['spent']-lastgain>maxplat[0]:maxplat=(rr[-1]['spent']-lastgain,lastgain,rr[-1]['spent'])
  # State BEFORE each action for fallback split.
  before=[(prev['best_validated']>0,bool(x.get('fallback'))) for prev,x in zip(rr,rr[1:])]
  row={'cell':str(key[:2]),'policy':key[2],'seed':seed,'spend':rr[-1]['spent'],'steps':cnt,'fallback':fallback,'fbshare':fallback/cnt,'fb_spend':fb_spend,'scaleups':scaleups,
       **counts,'new_lots':len(lots)-5,'new_builds':len(new),'validated':sum(b['J'] is not None for b in new),
       'no_T_or_I':sum(b['U'] is None and b['m'] is None for b in new),'no_T':sum(b['U'] is None for b in new),'no_I':sum(b['m'] is None for b in new),'imm_pass':sum(b['m'] is not None and b['m']>=r['m_ref'] for b in new),
       'dupl_lots':dup_lots,'dupl_builds':dup_builds,'dupl_utah':dup_u,'zero_spend_steps':sum(deltas==0),'maxplateau':maxplat[0],'cheap_streak':maxcheap[0],'cheap_streak_spend':maxcheap[1],
       'fb_prepositive':sum(f for pos,f in before if not pos),'n_prepositive':sum(not pos for pos,f in before),'fb_postpositive':sum(f for pos,f in before if pos),'n_postpositive':sum(pos for pos,f in before)}
  audit.append(row);details.append((key,seed,maxplat,maxcheap))
auditdf=pd.DataFrame(audit);auditdf.to_csv(out/'action_audit_per_seed.csv',index=False)
head_audit=auditdf.query("cell=='(1, 1)'")
cols=['spend','steps','fallback','fbshare','scaleups',*COST,'new_lots','validated','no_T_or_I','no_T','no_I','imm_pass','dupl_lots','dupl_builds','dupl_utah','zero_spend_steps','maxplateau','cheap_streak','cheap_streak_spend']
print(head_audit.groupby('policy')[cols].mean().to_string(float_format=lambda x:f'{x:.3f}'))
print('end spend min/max',head_audit.groupby('policy').spend.agg(['min','max']).to_string())
print('fallback per seed high',head_audit.query("policy=='pkgfn'")[['seed','steps','fallback','fbshare','fb_spend','validated','no_T_or_I','spend']].to_string(index=False))
print('fallback fractions and counts, action/spend')
for key,v in auditdf.groupby(['cell','policy']):
 if key[1].startswith('pkgfn'):
  print(key,'fb',v.fallback.sum(),'/',v.steps.sum(),v.fallback.sum()/v.steps.sum(),'spend',v.fb_spend.sum()/v.spend.sum(),'pre',v.fb_prepositive.sum(),v.n_prepositive.sum(),'post',v.fb_postpositive.sum(),v.n_postpositive.sum())
print('longest plateaus',head_audit.sort_values('maxplateau',ascending=False)[['policy','seed','maxplateau','cheap_streak','cheap_streak_spend']].head(15).to_string(index=False))
print('duplicates',auditdf.query('dupl_lots>0 or dupl_builds>0 or dupl_utah>0 or zero_spend_steps>0')[['cell','policy','seed','dupl_lots','dupl_builds','dupl_utah','zero_spend_steps']].to_string(index=False))


                  spend  steps  fallback  fbshare  scaleups       C      A      U      M      S      T      K      I  new_lots  validated  no_T_or_I   no_T   no_I  imm_pass  dupl_lots  dupl_builds  dupl_utah  zero_spend_steps  maxplateau  cheap_streak  cheap_streak_spend
policy                                                                                                                                                                                                                                                                         
ei            99947.000 21.000     0.000    0.000    21.000  19.700 21.000 21.000 21.000 21.000 21.000 21.000 21.000    21.000     21.000      0.000  0.000  0.000     1.550      0.050        0.000      0.000             0.000   42358.000         0.000               0.000
eifn          99960.000 21.000     0.000    0.000    21.000  21.000 21.000 21.000 21.000 21.000 21.000 21.000 21.000    21.000     21.000      0.000  0.000  0.000     7.500      0.000 

In [11]:
for base in ['tiered','tiered_ni_ref']:
 co=np.intersect1d(seedids[1,1,'pkgfn'],seedids[1,1,base]);a=curves[1,1,'pkgfn'][np.searchsorted(seedids[1,1,'pkgfn'],co)];b=curves[1,1,base][np.searchsorted(seedids[1,1,base],co)];t=b[:,-1].mean()
 print('matched ratio',base,'target',t,'baseline$',crossing(b.mean(0),t),'method$',crossing(a.mean(0),t))
# SE differences at the original presentation precision, including alternate metric.
# Independent rec_printed reconstruction: no running maximum, matching its recommendation interpretation.
def alt_curve(r,cell):
 rr=r['rows'];s=np.array([x['spent'] for x in rr]);v=np.array([x['rec_printed'] for x in rr]);i=np.searchsorted(s,GRID,'right')-1
 return np.where(i>=0,v[np.maximum(i,0)],np.nan)/OPT[cell]*100
sechange=[]
for metric in ['best_validated','rec_printed']:
 for key,rs in runs.items():
  if metric=='rec_printed' and key[:2]!=(1,1):continue
  a=curves[key] if metric=='best_validated' else np.array([alt_curve(r,key[:2]) for s,r in sorted(rs.items())])
  if metric=='rec_printed':assert np.allclose(a,ana['curves'](ana['load'](key[:2])[key[2]],key[:2],metric)*100)
  for bud in BUD:
   v=a[:,bud//250];old=v.std(ddof=0)/np.sqrt(len(v));new=st.sem(v) if len(v)>1 else np.nan
   if len(v)==1 or f'{old:.0f}'!=f'{new:.0f}':sechange.append({'metric':metric,'cell':str(key[:2]),'policy':key[2],'budget':bud,'old_exact':old,'new_exact':new,'printed_old':f'{old:.0f}','printed_new':f'{new:.0f}'})
print(pd.DataFrame(sechange).to_string(index=False));pd.DataFrame(sechange).to_csv(out/'SE_rounding_changes.csv',index=False)
# longest no-improvement spend blocks and counts.
plateau=[]
for key,rs in runs.items():
 for seed,r in rs.items():
  n=0;spend=0.;best=r['rows'][0]['best_validated'];maxs=0.;maxn=0
  for prev,x in zip(r['rows'],r['rows'][1:]):
   if x['best_validated']>best+1e-12:best=x['best_validated'];n=0;spend=0.
   else:n+=1;spend+=x['spent']-prev['spent'];maxs=max(maxs,spend);maxn=max(maxn,n)
  plateau.append({'cell':str(key[:2]),'policy':key[2],'seed':seed,'max_no_gain_spend':maxs,'max_no_gain_actions':maxn,'no_gain_run':abs(r['rows'][-1]['best_validated']-r['rows'][0]['best_validated'])<1e-12})
plateau=pd.DataFrame(plateau);print(plateau.query("cell=='(1, 1)'").groupby('policy').agg({'no_gain_run':'sum','max_no_gain_spend':['mean','max'],'max_no_gain_actions':'max'}).to_string())
print('high cheap worst',head_audit.query("policy=='pkgfn'").sort_values('cheap_streak',ascending=False)[['seed','cheap_streak','cheap_streak_spend']].to_string(index=False))
print('light cheap worst',head_audit.query("policy=='pkgfn_light'").sort_values('cheap_streak',ascending=False)[['seed','cheap_streak','cheap_streak_spend']].head(5).to_string(index=False))


matched ratio tiered target -10.434197542794156 baseline$ 98750.0 method$ 4750.0
matched ratio tiered_ni_ref target 27.814581422447475 baseline$ 82000.0 method$ 58250.0
        metric   cell      policy  budget  old_exact  new_exact printed_old printed_new
best_validated (0, 0)  eifn_reuse   25000   0.000000        NaN           0         nan
best_validated (0, 0)  eifn_reuse   50000   0.000000        NaN           0         nan
best_validated (0, 0)  eifn_reuse  100000   0.000000        NaN           0         nan
best_validated (0, 0)        eifn   25000   0.000000        NaN           0         nan
best_validated (0, 0)        eifn   50000   0.000000        NaN           0         nan
best_validated (0, 0)        eifn  100000   0.000000        NaN           0         nan
best_validated (1, 1)       pkgfn   50000  10.284898  11.108960          10          11
best_validated (1, 1)       pkgfn  100000  11.302330  12.207911          11          12
best_validated (1, 1)      random   250

In [12]:
# Uncertainty for paired endpoint differences (seed bootstrap, separate from crossing ratios).
rng=np.random.default_rng(20261027)
com=seedids[1,1,'pkgfn'];a=curves[1,1,'pkgfn'];l=curves[1,1,'pkgfn_light'][np.searchsorted(seedids[1,1,'pkgfn_light'],com)]
dv=a[:,-1]-l[:,-1];bd=dv[rng.integers(0,7,(50000,7))].mean(1)
print('high minus light endpoint mean, sampleSE,90%bootCI',dv.mean(),st.sem(dv),np.percentile(bd,[5,95]))
print('se matched light',[(b,l[:,b//250].std(ddof=0)/np.sqrt(7),st.sem(l[:,b//250])) for b in BUD])
for pol in POL:
 vv=list(runs[1,1,pol].values())
 print(pol,'settings variants',collections.Counter(tuple((k,str(r.get('settings',{}).get(k))) for k in ['S_ACQ','N_CAND','N_POOL','N_ANCHOR','N_PERTURB','CODE_VERSION','NOGATE']) for r in vv))
print('inventory all',len(list(Path('results').rglob('*.json'))),len(list(Path('quality/results-light').rglob('*.json'))),len(list(Path('compare/results-nogate').rglob('*.json'))))
# Check README and tables numeric rows that are generated by supplied script.
readme=Path('README.md').read_text();tables=Path('tables.md').read_text()
mdrows=collections.Counter(x for x in tables.splitlines() if x.startswith('|') and not x.startswith('|---'))
rrows=collections.Counter(x for x in readme.splitlines() if x.startswith('|') and not x.startswith('|---'))
print('exactly identical table lines reused in README',sum((mdrows&rrows).values()))
print('no gate first', {k:v for k,v in json.loads(next(Path('compare/results-nogate').rglob('*.json')).read_text()).items() if k not in ['rows','builds','lots']})


high minus light endpoint mean, sampleSE,90%bootCI -5.336754398093985 11.59941183691252 [-23.78386755  11.72120336]
se matched light [(25000, np.float64(10.193978114428742), np.float64(11.010754807476228)), (50000, np.float64(11.565453256810681), np.float64(12.492117269491118)), (100000, np.float64(4.3248977971753995), np.float64(4.671423528434851))]
pkgfn settings variants Counter({(('S_ACQ', '64'), ('N_CAND', '16'), ('N_POOL', '512'), ('N_ANCHOR', '[32, 16]'), ('N_PERTURB', '24'), ('CODE_VERSION', '2026-10-27'), ('NOGATE', 'False')): 7})
pkgfn_light settings variants Counter({(('S_ACQ', '32'), ('N_CAND', '10'), ('N_POOL', '256'), ('N_ANCHOR', 'None'), ('N_PERTURB', 'None'), ('CODE_VERSION', 'None'), ('NOGATE', 'None')): 20})
eifn settings variants Counter({(('S_ACQ', '32'), ('N_CAND', '10'), ('N_POOL', '256'), ('N_ANCHOR', 'None'), ('N_PERTURB', 'None'), ('CODE_VERSION', 'None'), ('NOGATE', 'None')): 20})
eifn_reuse settings variants Counter({(('S_ACQ', '32'), ('N_CAND', '10'), ('N_P

In [13]:
from scipy.special import expit
from scipy.optimize import differential_evolution, minimize
# Independent NumPy transcription for the OPTIONAL no-gate cross-check and oracle bounds.
def objective_np(x,kappa=1.,nogate=False):
 x=np.asarray(x);ni,al,fe,mn,a,P,v=np.moveaxis(x,-1,0)
 s=(1-np.exp(-(.6*ni+.9*al+3*ni*al))+.1*fe+.05*mn)/1.15
 f=np.clip(.65*ni+.3*mn-.35*al+2.5*ni*al+.05*fe,0,1)
 g=(.7*fe*(1+.5*al)+.15*al*ni)/1.2;h=.6*(1-ni)**2*(1-.6*al)
 q=np.exp(-((a-(.35+.4*f))/.28)**2)*(1-.35*f**2)*(1-.1*mn)
 window=np.exp(-((P-v-(.05+.25*f))/.2)**2)*(1-.35*expit((P-.85)/.05))
 crack=expit((f*(1-kappa*.85*g/(g+.12))-.56)/.05)
 rho=window*q**.3*(1-.75*crack)*(1-expit((h-P)/.08))
 beta=al**1.5*(1-.5*ni)*expit((v-P+.2)/.15)
 U=(.25+.75*s)*rho*(1+.15*beta)
 m=np.clip(.3+.55*(1-np.exp(-ni/.15))+.15*al-.6*al*beta+.3*np.clip(fe,0,.4)-.15*np.maximum(fe-.5,0)-.3*(1-rho),0,1)
 return U*(1. if nogate else expit((m-d['m_ref'])/.05))-.3*(ni+.3*al+.2*mn)
opt_nogate=[]
for seed in range(3):
 opt=differential_evolution(lambda x:-objective_np(x,nogate=True),[(0,1)]*7,seed=seed,popsize=15,tol=1e-10,maxiter=1200,polish=True,workers=1)
 opt_nogate.append((-opt.fun,opt.x,opt.success))
print('independently optimized no-gate values',opt_nogate)
ngopt=max(o[0] for o in opt_nogate)
ng=collections.defaultdict(dict)
for p in Path('compare/results-nogate').rglob('*.json'):
 r=json.loads(p.read_text());ng[r['kind']][r['seed']]=r
ngrows=[]
for pol,rs in sorted(ng.items()):
 a=[]
 for s,r in sorted(rs.items()):
  rr=r['rows'];spent=np.array([z['spent'] for z in rr]);v=np.maximum.accumulate([z['best_validated'] for z in rr]);i=np.searchsorted(spent,GRID,'right')-1;a.append(v[np.maximum(i,0)]/ngopt*100)
 a=np.array(a)
 row={'policy':pol,'n':len(a),'start':a[:,0].mean()}
 for b in BUD:row[f'mean{b}']=a[:,b//250].mean();row[f'se{b}']=st.sem(a[:,b//250]);row[f'published_se{b}']=a[:,b//250].std()/np.sqrt(len(a))
 ngrows.append(row)
ngdf=pd.DataFrame(ngrows);print(ngdf.to_string(index=False,float_format=lambda x:f'{x:.5f}'));ngdf.to_csv(out/'nogate_summary.csv',index=False)
# Random rescoring uses IDENTICAL serialized designs; check all 26 each seed.
random_equal=True
for s,r in ng['random'].items():
 orig=runs[1,1,'random'][s]
 random_equal &= [(b['c'],b['a'],b['pv']) for b in r['builds']]==[(b['c'],b['a'],b['pv']) for b in orig['builds']]
print('random no gate identical designs',random_equal)
# All-deposited oracle objective (rounded compositions make this approximate).
for pol in ['pkgfn','pkgfn_light','eifn']:
 oracle=[]
 for s,r in runs[1,1,pol].items():
  xx=np.array([b['c']+[b['a']]+b['pv'] for b in r['builds']]);oracle.append(objective_np(xx).max()/OPT[1,1]*100)
 print('oracle all deposited',pol,np.mean(oracle),'validated mean',curves[1,1,pol][:,-1].mean())


independently optimized no-gate values [(np.float64(0.7450514788538778), array([1.61251403e-01, 1.00000000e+00, 1.00000000e+00, 2.94151103e-11,
       4.33177212e-01, 6.06112855e-01, 5.07889184e-01]), True), (np.float64(0.745051478834349), array([1.61253320e-01, 1.00000000e+00, 1.00000000e+00, 1.51255120e-11,
       4.33179863e-01, 6.06109368e-01, 5.07884272e-01]), True), (np.float64(0.7450514788314337), array([1.61252742e-01, 1.00000000e+00, 1.00000000e+00, 4.19226462e-11,
       4.33177075e-01, 6.06108831e-01, 5.07884171e-01]), True)]
    policy  n    start  mean25000  se25000  published_se25000  mean50000  se50000  published_se50000  mean100000  se100000  published_se100000
        ei 20 22.51825   38.94622  5.95415            5.80339   48.80072  5.52940            5.38939    65.15631   4.61025             4.49351
      eifn 20 22.51825   43.72614  6.11034            5.95562   66.78169  5.14828            5.01792    84.77406   4.85349             4.73059
eifn_reuse 20 22.51825   47.

In [14]:
from PIL import Image
for name in ['bofn_cu_best_vs_spend_k1_partial_vs_full.png','bofn_cu_best_vs_spend_k1_partial_vs_full_current.png']:
 im=Image.open(name);print(name,im.size,im.info)
print('font availability',Path.home(),list((Path.home()/'.local/share/fonts/aptos').glob('*.ttf'))[:3])
# locate axes/colored lines in the actual raster; original PNG uses light pKGFN.
im=np.array(Image.open('bofn_cu_best_vs_spend_k1_partial_vs_full.png').convert('RGB'))
print('dimensions',im.shape)
# infer graph x coordinates from 0/100 bottom axis, long exact-gray horizontal segment.
mask=np.max(abs(im.astype(int)-np.array([140,140,140])),axis=2)<=2
ys=np.flatnonzero(mask.sum(axis=1)>im.shape[1]*.40)
print('long axis y',ys[:30],ys[-30:])
for y in ys[-3:]:
 xs=np.flatnonzero(mask[y]);print(y,xs.min(),xs.max(),len(xs))


bofn_cu_best_vs_spend_k1_partial_vs_full.png (3300, 1860) {'Software': 'Matplotlib version3.11.2, https://matplotlib.org/', 'dpi': (299.9994, 299.9994)}
bofn_cu_best_vs_spend_k1_partial_vs_full_current.png (3300, 1860) {'Software': 'Matplotlib version3.11.2, https://matplotlib.org/', 'dpi': (299.9994, 299.9994)}
font availability /home/kerneluser []
dimensions (1860, 3300, 3)


long axis y [1518 1519 1520] [1518 1519 1520]
1518 362 2113 1752
1519 362 2113 1752
1520 362 2113 1752


In [15]:
colors={'pkgfn_light':(192,0,0),'eifn':(64,64,64),'ei':(180,180,180),'tiered':(140,140,140)}
start=curves[1,1,'eifn'].mean(0)[0];top=curves[1,1,'eifn'].mean(0)[-1]
ylo=start-9;yhi=top+4;yb=1860*(1-.21);yt=1860*(1-.86)
def ypct(y):return ylo+(yb-y)/(yb-yt)*(yhi-ylo)
print('transform',ylo,yhi,yb,yt)
read_off=[]
for bud in BUD:
 xx=int(round(3300*.11+(3300*.64-3300*.11)*bud/100000))
 for pol,color in colors.items():
  expected=curves[1,1,pol][:,bud//250].mean()
  yexp=yb-(expected-ylo)/(yhi-ylo)*(yb-yt)
  crop=im[max(0,int(yexp)-15):int(yexp)+16,max(0,xx-1):xx+2,:]
  hits=np.where(np.max(abs(crop.astype(int)-np.array(color)),axis=2)<=2)[0]
  val=ypct(max(0,int(yexp)-15)+np.median(hits)) if len(hits) else np.nan
  read_off.append({'policy':pol,'budget':bud,'read_off':val,'curve':expected,'x_pixel':xx,'pixel_hits':len(hits)})
print(pd.DataFrame(read_off).to_string(index=False,float_format=lambda x:f'{x:.3f}'))
pd.DataFrame(read_off).to_csv(out/'figure_raster_readings.csv',index=False)


transform -18.60955489703411 77.21174313092199 1469.4 260.40000000000003
     policy  budget  read_off  curve  x_pixel  pixel_hits
pkgfn_light   25000    12.015 11.951      800          39
       eifn   25000       NaN 14.977      800           0
         ei   25000     5.477  5.680      800           6
     tiered   25000    -9.384 -9.427      800          27
pkgfn_light   50000    27.232 27.440     1238          58
       eifn   50000       NaN 38.774     1238           0
         ei   50000     7.220  7.180     1238          30
     tiered   50000    -9.067 -9.106     1238          27
pkgfn_light  100000    58.261 58.552     2112          22
       eifn  100000       NaN 73.212     2112           0
         ei  100000    14.036 14.421     2112          20
     tiered  100000    -7.640 -7.691     2112           9


In [16]:
for pol in ['pkgfn','pkgfn_light']:
 rr=list(runs[1,1,pol].values());print('wall',pol,'all mean min max mins',np.mean([r['wall_s'] for r in rr])/60,min(r['wall_s'] for r in rr)/60,max(r['wall_s'] for r in rr)/60)
 rr=[runs[1,1,pol][int(s)] for s in com];print('matched wall mean',np.mean([r['wall_s'] for r in rr])/60)
# Fix raster dark line detection: red uncertainty bands tint the otherwise gray BOFN line.
for bud in BUD:
 xx=int(round(3300*.11+(3300*.64-3300*.11)*bud/100000));expected=curves[1,1,'eifn'][:,bud//250].mean();yexp=yb-(expected-ylo)/(yhi-ylo)*(yb-yt)
 crop=im[int(yexp)-20:int(yexp)+21,xx-1:xx+2,:];hits=np.where((crop[:,:,0]<110)&(crop[:,:,1]<100)&(crop[:,:,2]<100))[0]
 print('BOFN raster',bud,ypct(int(yexp)-20+np.median(hits)) if len(hits) else None)
# Per-policy means and max uninterrupted cheap actions by definition <= $500 per action.
print('worst high plateau',head_audit.query("policy=='pkgfn'").sort_values('maxplateau',ascending=False)[['seed','maxplateau']].to_string(index=False))
# First strict-positive medians on the actual grid and exact rows (README rounds them).
for pol in POL:
 c=curves[1,1,pol];costs=np.array([GRID[np.flatnonzero(row>0)[0]] if np.any(row>0) else np.inf for row in c]);print('first>0',pol,np.median(costs),sum(np.isfinite(costs)),sum(c[:,0]>0))


wall pkgfn all mean min max mins 62.39207427899043 32.65662939548493 83.50889884233474
matched wall mean 62.39207427899043
wall pkgfn_light all mean min max mins 9.294701427022614 6.518983813126882 12.716295369466145
matched wall mean 11.227966862633114
BOFN raster 25000 15.14585521885217
BOFN raster 50000 38.80396726297532
BOFN raster 100000 73.08247131350552
worst high plateau  seed  maxplateau
    2     77900.0
    5     68440.0
    3     58730.0
    4     52960.0
    1     50250.0
    7     44000.0
    0     41790.0
first>0 pkgfn 47250.0 6 2
first>0 pkgfn_light 38875.0 19 5
first>0 eifn 14500.0 20 5
first>0 eifn_reuse 21375.0 20 5
first>0 tiered inf 5 5
first>0 tiered_ni 43000.0 13 5
first>0 tiered_ni_ref 19125.0 17 5
first>0 ei 24000.0 20 5
first>0 random 31125.0 17 5


In [17]:
# Assemble machine-readable exact SE audit, including all nonzero differences, not just rounded changes.
se_all=[]
for metric in ['best_validated','rec_printed']:
 for key,rs in runs.items():
  if metric=='rec_printed' and key[:2]!=(1,1):continue
  a=curves[key] if metric=='best_validated' else np.array([alt_curve(r,key[:2]) for s,r in sorted(rs.items())])
  for bud in BUD:
   v=a[:,bud//250];se_all.append({'metric':metric,'cell':str(key[:2]),'policy':key[2],'n':len(v),'spend':bud,'mean':v.mean(),'supplied_SE':v.std()/np.sqrt(len(v)),'sample_SE':st.sem(v) if len(v)>1 else np.nan})
for pol in ['pkgfn_light','eifn','eifn_reuse','tiered_ni_ref']:
 a=curves[1,1,pol][np.searchsorted(seedids[1,1,pol],seedids[1,1,'pkgfn'])]
 for bud in BUD:
  v=a[:,bud//250];se_all.append({'metric':'best_validated_matched7','cell':'(1, 1)','policy':pol,'n':len(v),'spend':bud,'mean':v.mean(),'supplied_SE':v.std()/np.sqrt(len(v)),'sample_SE':st.sem(v)})
pd.DataFrame(se_all).to_csv(out/'all_SE_comparisons.csv',index=False)
# Save exact action tally and quality/fidelity checks as JSON for reproducibility.
quality=[]
for cell in [(1,1),(0,0)]:
 rs=[r for key,dd in runs.items() if key[:2]==cell for r in dd.values()]
 v=auditdf.query('cell==@str(cell)') if False else auditdf[auditdf.cell==str(cell)]
 kg=v[v.policy.str.startswith('pkgfn')]
 quality.append({'cell':cell,'runs':len(rs),'gp_fits':sum(r['gp_fits']['calls'] for r in rs),'fit_failures':sum(r['gp_fits']['failures'] for r in rs),'fallback':int(kg.fallback.sum()),'purchases':int(kg.steps.sum())})
print('quality',quality)
print('high wall per seed',[(s,round(r['wall_s']/60,3)) for s,r in sorted(runs[1,1,'pkgfn'].items())])
print('light matched fallback fraction',sum(r.get('fallback',False) for s in seedids[1,1,'pkgfn'] for r in runs[1,1,'pkgfn_light'][int(s)]['rows'][1:])/sum(len(runs[1,1,'pkgfn_light'][int(s)]['rows'])-1 for s in seedids[1,1,'pkgfn']))
# Record longest cheap stretches with exact action list and path.
cheap_records=[]
for key,seed,plat,cheap in details:
 if key[:2]==(1,1) and key[2] in ['pkgfn','pkgfn_light']:
  cheap_records.append({'policy':key[2],'seed':seed,'count':cheap[0],'spend':cheap[1],'start':cheap[2],'end':cheap[3]})
print('cheap_records extremes',sorted(cheap_records,key=lambda x:x['count'],reverse=True)[:5])
(out/'quality_checks.json').write_text(json.dumps(quality,indent=2))
# Persist only project workflow conventions, not results.
mem=Path('.prompts/MEMORY.md').read_text().replace('[Fill me in]', 'Copper benchmark independent-review workflow.',1).replace('[Fill me in]', '- Read settings.md before analysis; write independent NumPy/SciPy reconstruction before inspecting supplied aggregation.\n- Deliver a Markdown report both in the answer and at outputs/report.md, with severity-ranked discrepancies, side-by-side tables, and executed-versus-inspected checks.\n- Keep original inputs unchanged; save reproducibility code and numerical audit files under outputs/.',1)
Path('.prompts/MEMORY.md').write_text(mem)


quality [{'cell': (1, 1), 'runs': 167, 'gp_fits': 75434, 'fit_failures': 0, 'fallback': 685, 'purchases': 2633}, {'cell': (0, 0), 'runs': 12, 'gp_fits': 1919, 'fit_failures': 0, 'fallback': 424, 'purchases': 952}]
high wall per seed [(0, 81.988), (1, 78.321), (2, 83.509), (3, 81.623), (4, 43.584), (5, 35.063), (7, 32.657)]
light matched fallback fraction 0.2743988684582744
cheap_records extremes [{'policy': 'pkgfn_light', 'seed': 15, 'count': 20, 'spend': np.float64(5570.0), 'start': 27920.0, 'end': 33490.0}, {'policy': 'pkgfn_light', 'seed': 14, 'count': 16, 'spend': np.float64(7900.0), 'start': 50230.0, 'end': 58130.0}, {'policy': 'pkgfn_light', 'seed': 6, 'count': 14, 'spend': np.float64(5720.0), 'start': 11050.0, 'end': 16770.0}, {'policy': 'pkgfn', 'seed': 5, 'count': 10, 'spend': np.float64(3510.0), 'start': 33850.0, 'end': 37360.0}, {'policy': 'pkgfn_light', 'seed': 1, 'count': 10, 'spend': np.float64(100.0), 'start': 90650.0, 'end': 90750.0}]


783

In [18]:
# Supplemental checks for README claims and table omissions.
print('no-gain run counts strict vs numerical tolerances')
for pol in ['ei','random','tiered','tiered_ni','tiered_ni_ref']:
 vv=list(runs[1,1,pol].values());print(pol,sum(r['rows'][-1]['best_validated']==r['rows'][0]['best_validated'] for r in vv),sum(abs(r['rows'][-1]['best_validated']-r['rows'][0]['best_validated'])<1e-10 for r in vv))
# infer stage use from inventory and compare to purchase table rounded cells, independently.
for pol in POL:
 rs=list(runs[1,1,pol].values())
 assert all(len(r['lots'])-5==r['counts']['A'] for r in rs)
 print(pol,'mean sunk-inclusive seeds valid',sum(all(b[f] is not None for f in ['U','m','J']) for r in rs for b in r['builds'][:5]))
# Spot exact duplicate lot in standard BO seed 19.
r=runs[1,1,'ei'][19];cc=collections.defaultdict(list)
for i,l in enumerate(r['lots']):cc[tuple(l['c'])+(l['a'],)].append(i)
print('duplicate lot seed19',[(k,v) for k,v in cc.items() if len(v)>1])
# Fixed target ratios reuse relative to full BOFN (not to tiered, which never reaches).
for t in [55,60,65]:
 f=crossing(curves[1,1,'eifn'].mean(0),t);re=crossing(curves[1,1,'eifn_reuse'].mean(0),t)
 print('full/reuse fixed',t,f/re)
# Current plot's huge last-grid rise.
print('pkgfn mean 99.75k ->100k',curves[1,1,'pkgfn'][:,-2:].mean(0))
print('light mean 99.75k ->100k',curves[1,1,'pkgfn_light'][:,-2:].mean(0))
# Read off dashed reuse using nearby red pixels near expected mean; approximate only.
for bud in BUD:
 xx=int(round(3300*.11+(3300*.64-3300*.11)*bud/100000));expected=curves[1,1,'eifn_reuse'][:,bud//250].mean();yexp=yb-(expected-ylo)/(yhi-ylo)*(yb-yt)
 crop=im[int(yexp)-12:int(yexp)+13,xx-10:xx+11,:];hits=np.where((crop[:,:,0]>140)&(crop[:,:,0]<220)&(crop[:,:,1]<20)&(crop[:,:,2]<20))[0]
 print('Reuse raster',bud,ypct(int(yexp)-12+np.median(hits)) if len(hits) else None,'expected',expected)


no-gain run counts strict vs numerical tolerances
ei 0 0
random 4 4
tiered 15 15
tiered_ni 11 11
tiered_ni_ref 3 3
pkgfn mean sunk-inclusive seeds valid 35
pkgfn_light mean sunk-inclusive seeds valid 100
eifn mean sunk-inclusive seeds valid 100
eifn_reuse mean sunk-inclusive seeds valid 100
tiered mean sunk-inclusive seeds valid 100
tiered_ni mean sunk-inclusive seeds valid 100
tiered_ni_ref mean sunk-inclusive seeds valid 100
ei mean sunk-inclusive seeds valid 100
random mean sunk-inclusive seeds valid 100
duplicate lot seed19 [((0.0, 0.0, 1.0, 0.0, 0.0), [14, 20])]
full/reuse fixed 55 1.1596958174904943
full/reuse fixed 60 1.2077464788732395
full/reuse fixed 65 1.058641975308642
pkgfn mean 99.75k ->100k [46.21409869 58.85574037]
light mean 99.75k ->100k [53.34305277 58.55212235]
Reuse raster 25000 14.947713576941588 expected 14.3525437184516
Reuse raster 50000 32.304921408308324 expected 32.27422874692295
Reuse raster 100000 72.01250644718837 expected 72.03264442442


In [19]:
# Independent scale-up accounting via sessions-used and final per-lot inventory.
# Seed lots start with 3 unused sessions after their sunk seed build; new lots start at 0.
ledger=[]
for key,rs in runs.items():
 for seed,r in rs.items():
  initial_capacity=5*3
  capacity_left=sum(l['sessions_left'] for l in r['lots'])
  purchased_scaleups=(r['counts']['M']+capacity_left-initial_capacity)/4
  predicted=sum(r['counts'][k]*v for k,v in COST.items())+1600*purchased_scaleups
  assert purchased_scaleups>=0 and purchased_scaleups==int(purchased_scaleups)
  assert predicted==r['rows'][-1]['spent'],(key,seed,predicted)
  ledger.append({'path':r['_path'],'seed':seed,'policy':key[2],'scaleups_inventory':purchased_scaleups,'predicted_spend':predicted,'logged_spend':r['rows'][-1]['spent']})
pd.DataFrame(ledger).to_csv(out/'inventory_cost_ledger.csv',index=False)
print('Inventory cost reconciliations',len(ledger),'all exact')
print('stored best_validated decreases',sum(np.any(np.diff([x['best_validated'] for x in r['rows']])<0) for rs in runs.values() for r in rs.values()))
for pol in POL:
 for s,r in runs[1,1,pol].items():
  assert r['rows'][0]['best_validated']==runs[1,1,'eifn'][s]['rows'][0]['best_validated']
print('Initial values identical across all common seeds')


Inventory cost reconciliations 179 all exact
stored best_validated decreases 0
Initial values identical across all common seeds


In [20]:
# Construct compact side-by-side report tables directly from executed results.
labels={'pkgfn':'pKGFN, production','pkgfn_light':'pKGFN, light','eifn':'BOFN, full','eifn_reuse':'BOFN + reuse','tiered':'Tiered, spec','tiered_ni':'Tiered, Ni floor','tiered_ni_ref':'Tiered, Ni + reference','ei':'Standard BO','random':'Random'}
def mdtable(headers,rows):
 return '| '+' | '.join(headers)+' |\n|'+'|'.join(['---']*len(headers))+'|\n'+'\n'.join('| '+' | '.join(str(v) for v in row)+' |' for row in rows)
side=[]
for pol in POL:
 v=summary[(summary.cell=='(1, 1)')&(summary.policy==pol)].iloc[0]
 a=curves[1,1,pol]
 row=[labels[pol],len(a)]
 for b in BUD:
  x=a[:,b//250];row += [f'{x.mean():.3f} ± {st.sem(x):.3f}',f'{x.mean():.0f} ± {x.std()/np.sqrt(len(x)):.0f}']
 side.append(row)
main_table=mdtable(['Policy','n','My $25k','Supplied $25k','My $50k','Supplied $50k','My $100k','Supplied $100k'],side)
medrows=[]
for pol in POL:
 a=curves[1,1,pol];row=[labels[pol]]
 for t in [60,70,80]:
  ct=np.array([crossing(c,t) for c in a]);v=np.median(ct)
  my='not reached' if not np.isfinite(v) else f'${v:,.0f}'
  supplied=ana['fmt_cost'](v)
  row.append(f'{my} / {supplied}; {np.isfinite(ct).sum()}/{len(ct)}')
 medrows.append(row)
med_table=mdtable(['Policy','60%: mine / supplied; reached','70%: mine / supplied; reached','80%: mine / supplied; reached'],medrows)
def rat_table(base):
 rows=[]
 for pol in POL:
  if pol==base:continue
  r=ratio_results[(ratio_results.baseline==base)&(ratio_results.policy==pol)].iloc[0]
  b=exactboot[(exactboot.base==base)&(exactboot.policy==pol)].iloc[0]
  rows.append([labels[pol],'not reached' if not np.isfinite(r.method_cost) else f'${r.method_cost:,.0f}',f'{b.est:.4f}',f'{b.lo:.4f}–{b.hi:.4f}',f'{b.mlo:.4f}–{b.mhi:.4f}',f'{b.p3:.4f}'])
 return mdtable(['Method','First mean crossing','Ratio','90% independent CI','90% matched CI','Bootstrap fraction ≥3'],rows)
wrows=[]
for pol in POL:
 if pol=='pkgfn':continue
 row=[labels[pol]]
 for b in BUD:
  r=paired_results[(paired_results.against==pol)&(paired_results.budget==b)].iloc[0]
  row.append(f'{r.difference:+.3f}; {r.wins}/{r.ties}/{r.losses}; {r.p_scipy_default:.6f}')
 wrows.append(row)
wtable=mdtable(['Production pKGFN minus… (n=7)','$25k: Δ; W/T/L; p','$50k: Δ; W/T/L; p','$100k: Δ; W/T/L; p'],wrows)
purchase_rows=[]
for pol in POL:
 v=head_audit[head_audit.policy==pol]
 purchase_rows.append([labels[pol],*[f'{v[k].mean():.2f}' for k in COST],f'{v.validated.mean():.2f}',f'{v.scaleups.mean():.2f}'])
purchase_table=mdtable(['Policy',*COST,'Validated builds','Paid scale-ups'],purchase_rows)
fixed_table=mdtable(['Policy','55%','60%','65%'],[[labels[r.policy],*[('not reached' if not np.isfinite(r[f'cost{t}']) else f'${r[f"cost{t}"]:,.0f}') for t in [55,60,65]]] for _,r in targetdf.iterrows()])
fallback_table=mdtable(['Seed','Fallback actions / all','Share','Spend on fallback','New validated / deposited'],[[int(r.seed),f'{int(r.fallback)}/{int(r.steps)}',f'{100*r.fbshare:.1f}%',f'${r.fb_spend:,.0f}',f'{int(r.validated)}/{int(r.M)}'] for _,r in head_audit[head_audit.policy=='pkgfn'].iterrows()])
sensrows=[]
for pol in ['pkgfn_light','eifn','eifn_reuse']:
 r=summary[(summary.cell=='(0, 0)')&(summary.policy==pol)].iloc[0]
 sensrows.append([labels[pol],int(r.n),*[f'{r[f"mean{b}"]:.3f} ± '+('not estimable' if r.n==1 else f'{r[f"se{b}"]:.3f}') for b in BUD]])
sens_table=mdtable(['Cell (0,0) policy','n','$25k','$50k','$100k'],sensrows)
ng_table=mdtable(['No-gate policy','n','$25k','$50k','$100k'],[[labels[r.policy],int(r.n),*[f'{r[f"mean{b}"]:.3f} ± {r[f"se{b}"]:.3f}' for b in BUD]] for _,r in ngdf.iterrows()])
print(main_table)


| Policy | n | My $25k | Supplied $25k | My $50k | Supplied $50k | My $100k | Supplied $100k |
|---|---|---|---|---|---|---|---|
| pKGFN, production | 7 | 3.341 ± 5.433 | 3 ± 5 | 24.572 ± 11.109 | 25 ± 10 | 58.856 ± 12.208 | 59 ± 11 |
| pKGFN, light | 20 | 11.951 ± 6.559 | 12 ± 6 | 27.440 ± 7.385 | 27 ± 7 | 58.552 ± 4.844 | 59 ± 5 |
| BOFN, full | 20 | 14.977 ± 5.745 | 15 ± 6 | 38.774 ± 9.116 | 39 ± 9 | 73.212 ± 7.322 | 73 ± 7 |
| BOFN + reuse | 20 | 14.353 ± 4.702 | 14 ± 5 | 32.274 ± 7.805 | 32 ± 8 | 72.033 ± 7.046 | 72 ± 7 |
| Tiered, spec | 20 | -9.427 ± 5.500 | -9 ± 5 | -9.106 ± 5.473 | -9 ± 5 | -7.691 ± 5.366 | -8 ± 5 |
| Tiered, Ni floor | 20 | -6.644 ± 5.497 | -7 ± 5 | 0.762 ± 5.537 | 1 ± 5 | 10.217 ± 6.382 | 10 ± 6 |
| Tiered, Ni + reference | 20 | 7.858 ± 4.225 | 8 ± 4 | 16.136 ± 3.960 | 16 ± 4 | 24.233 ± 4.750 | 24 ± 5 |
| Standard BO | 20 | 5.680 ± 3.413 | 6 ± 3 | 7.180 ± 3.299 | 7 ± 3 | 14.421 ± 6.256 | 14 ± 6 |
| Random | 20 | -0.612 ± 4.532 | -1 ± 4 | 6.904 ± 4.977 | 7 ± 

In [21]:
findings_rows=[
['Changes the conclusion','F1 — Claim scope','Network-aware BOFN beats the baselines here; choosing stages does not demonstrably beat full-pipeline BOFN. Production pKGFN is 58.856%, versus 73.212% full BOFN; matched difference −28.511 points, nominal p=0.046875, not multiplicity-robust.'],
['Changes the conclusion','F2 — Fidelity inference','“The lighter settings were not why pKGFN lags” is too categorical: only 7 matched seeds; high−light = −5.337 points, 90% paired-bootstrap interval −23.784 to +11.721. Fidelity, F2 anchoring and F4 fallback fixes changed together.'],
['Changes the conclusion','F3 — Matched-bootstrap estimand','The production pKGFN 2.1164× point uses 7 method/20 reference seeds; its matched interval 0.8086–1.9879 uses 7/7 and a different target. The proper matched point is 1.4077×. These are not two intervals for the same sample estimator.'],
['Changes the conclusion','F4 — Spec-tiered speed-up','The 17–20× ratios measure crossing a negative −7.691% level, not reaching a good alloy. The bootstrap upper limit 397× is an artifact of flooring method crossing spend at $250; zero-spend crossings really occur.'],
['Changes the conclusion','F5 — Causal/mechanistic wording','No-gate ablation supports gate sensitivity, not the exclusive claim “the reason is clear” or proof of what the learned model inferred. Non-random policies change their designs; no physical calibration or gate-bias factorial is available.'],
['Changes a number','F6 — Standard errors','All nonzero reported SEs use population SD (ddof=0), not sample SD. They are smaller by factors √((n−1)/n); n=1 gives a misleading ±0. Production pKGFN should be ±11 at $50k and ±12 at $100k, not ±10 and ±11.'],
['Changes a number','F7 — Runtime claim','README says production pKGFN takes ~25–35 min/run, 2.9× light. Stored wall_s: mean 62.39 min, range 32.66–83.51; matched light mean 11.23 min (ratio 5.56). Hardware/load equivalence is unknown.'],
['Changes a number','F8 — Omitted speed-up row','tiered_ni_ref is present but excluded from the speedup_table default kinds. Against spec tiered it is 11.0278×, $9,000, independent CI 5.0253–397, matched CI 7.2182–14.1786.'],
['Changes a number','F9 — Fragile final-budget pKGFN level','Production pKGFN mean jumps from 46.214% at $99,750 to 58.856% at $100,000; three late validations explain 12.642 points. This is legitimate step-function accounting, but the endpoint is unusually sensitive.'],
['Changes a number','F10 — Rounding/small numerical differences','Purchases and medians reproduce at the printed precision. Examples: 37.857 Mines sessions → 37.9; tiered_ni immersions 2.550 prints 2.5 in supplied floating-point formatting; reference passes 2.650 prints 2.6. Do not interpret rounded tenths as exact.'],
['Changes the conclusion','F11 — Incomplete sensitivities','No (1,0)/(0,1) runs; (0,0) has light pKGFN n=10 and BOFN variants n=1. No screen/audit/mines_only/funnel results. These cannot establish robustness, a screen-audit benefit or a two-step DED benefit.'],
['Cosmetic / potentially misleading','F12 — Figure labeling and uncertainty','Proposal PNG is light pKGFN, not production. “Partial: BOFN + reuse” means full downstream testing with lot reuse, not free stage selection. Bands are selective, unlabeled ±SE; full BOFN and standard BO have no bands.'],
['Cosmetic','F13 — Small-figure readability','Only endpoint axis ticks; no zero line or intermediate spend ticks. At 3.1 in, 24-point source labels scale to ~6.76 pt; most width is used by direct labels, leaving ~1.64 in of data axes.'],
['Verified, no numerical discrepancy','F14 — Internal consistency','Every reconstructed best_validated curve equals the supplied curve exactly; regenerating tables.md gives zero differing lines. Means, median thresholds, listed ratios, bootstrap quantiles, Wilcoxon results and purchase counts reproduce under the supplied conventions.']]
finding_table=mdtable(['Severity','Finding','Evidence / correction'],findings_rows)


In [22]:
report = f'''# Independent review of the copper function-network benchmark (uploaded snapshot)

**Verdict.** I independently reconstructed the stored runs before examining `analyze.py`. At the headline (κ,η)=(1,1), full-pipeline Bayesian optimization on the function network (BOFN) reaches **73.212%** of the stored optimum after $100,000, BOFN with lot reuse **72.033%**, corrosion-informed tiered screening (`tiered_ni_ref`) **24.233%**, standard Bayesian optimization **14.421%**, and production-setting partial knowledge-gradient function-network optimization (pKGFN) **58.856% on only seven completed seeds**. The comparison supports BOFN over these baselines *in this synthetic benchmark*; it does **not** show that deciding which stage to buy beats running a full chain. The README's integer-valued mean claims reproduce, but some standard errors, bootstrap interpretation, and causal/fidelity wording need correction. I did not run new Bayesian-optimization searches.

**Data and method.** I read `settings.md` first, then wrote and executed the independent NumPy/SciPy reconstruction in `outputs/independent_core.py` before opening `analyze.py`. I read 149 `results/` JSONs (seven production pKGFN and eight other 20-seed headline policies; two additional single-seed sensitivity runs), 30 light pKGFN JSONs (20 headline, ten sensitivity), and separately 80 no-gate JSONs. I use the exact (1,1) optimum **0.492187251856002** from `truth_tables.json`; the 0–$100,000 grid has $250 steps. Each seed contributes its last row with `spent <= grid point` (including the $0 shared seed row), running maximum of `best_validated`, divided by its cell optimum. Endpoints are arithmetic means; my **standard errors use sample SD/√n**. A missing threshold crossing is `∞`, not a fabricated $100k crossing. I leave all supplied inputs unchanged.

## Severity-ranked discrepancies and findings

{finding_table}

The production run provenance is confirmed from the seven JSON settings: S=64, 16 stage candidates, 512 pool, 32+16 anchors, 24 perturbations and `CODE_VERSION=2026-10-27`. Light files record S=32, 10 candidates, 256 pool; some old-file settings fields do not record the anchor/perturbation counts or code version. Source documentation says F2/F4 and settings changed together. The no-gate 80 JSONs are independent, normalized by an independently optimized no-gate truth (~0.74505148); I did not rely on a missing `compare/landscape.json`.

## 1. Headline values, side by side — **executed**

All entries below are percentage points of the true optimum; each `±` is a standard error. “Supplied” is the rounded value in both `tables.md` and README's headline tables (its SE uses population SD). These are **all nine policies actually present** at (1,1).

{main_table}

**Every 1-point displayed SE discrepancy in the headline table:** production pKGFN $50k 10→11 and $100k 11→12; light pKGFN $25k 6→7; tiered spec $25k 5→6; Ni-floor tiered $50k 5→6; random $25k 4→5. Means are unchanged at supplied precision. All unrounded SEs differ whenever within-policy variance is nonzero: multiply published SE by √(7/6)=1.0801 for production pKGFN or √(20/19)=1.0260 for 20-seed policies. The (0,0) light pKGFN $25k SE changes 4→5, while its two single-seed BOFN results should not display “± 0”. The README's alternate `rec_printed` table likewise has corrected displayed SE: production pKGFN $25k 10→11, $50k 12→13, $100k 6→7; light pKGFN $100k 4→5; random $25k 4→5. Every unrounded mean and SE is in `outputs/all_SE_comparisons.csv`.

Medians of **per-seed** first grid crossings (mine / supplied rounded notation; fraction reaching threshold by $100k) also agree exactly with the supplied integer-$k displays:

{med_table}

In particular, production pKGFN's 60% median is an exact grid crossing at **$100,000** (four of seven seeds reached 60%), not an extrapolation; the three others did not cross. Some 70% and 80% medians are `∞` because at least half the seeds did not reach the level. README lines 8–30 and 66–107 reproduce these rounded means, medians and paired tests under their own SE convention; its 49-point BOFN-versus-reference difference is **48.979** points and its 17/20 matched wins give a two-sided Wilcoxon **p=0.000322**. Its qualitative claim that the production pKGFN equals light pKGFN at 59% only concerns *unmatched, differently sized samples*.

### Speed-ups — **executed**

A speed-up is the baseline's **first mean-curve crossing of its own $100k final mean**, divided by the method's first mean-curve crossing of that same target. Each row below reproduces `tables.md`'s exact implementation, including its 2,000-replicate, seed-0, policy-independent and matched-seed percentile bootstraps; decimals expose differences hidden by the published one-decimal display. A non-crossing method is represented by **0×** in that implementation, rather than a measured speed-up. The extra `tiered_ni_ref` row versus spec tiered was omitted from the supplied table even though it exists in the files.

**Baseline: spec tiered**, final target **−7.690822%**, baseline first crossing **$99,250** (the supplied `$99k`):

{rat_table('tiered')}

**Baseline: corrosion-informed `tiered_ni_ref`**, final target **24.233129%**, baseline first crossing **$100,000**:

{rat_table('tiered_ni_ref')}

The published one-decimal CIs and probabilities in `tables.md` and the README agree with the exact rows above (subject to ordinary rounding). Most important: full BOFN needs **$33,500**, giving **2.9851×** (published 3.0×, independent 90% CI **1.6560–4.1667**, matched CI **1.6571–3.4783**); reuse needs **$35,250**, **2.8369×** (independent **1.6571–4.0941**, matched **1.6812–3.9216**); production pKGFN needs **$47,250**, **2.1164×** (independent **0.8766–2.4242**, matched **0.8086–1.9879**). The matched production pKGFN/reference bootstrap **instead uses baseline seeds 0,1,2,3,4,5,7**, whose baseline final mean is **27.8146%**; on that matched subset the point ratio is **$82,000/$58,250 = 1.4077×**. Label the matched point alongside its matched interval; the mixed-sample 2.1× should not be presented as the center of the 0.8–2.0 interval. The independent 90% interval includes 1 for production pKGFN, so its spend advantage is not established at that confidence level. A 90% CI does not prove a **≥3×** effect for full BOFN: its independent bootstrap fraction ≥3 is **0.156**. The production pKGFN bootstrap fraction ≥3 is **0.0065**.

The spec-tiered target is **below zero** and gives essentially every model-based policy a large ratio, even standard BO (19.85×) and `tiered_ni_ref` (11.03×); it is not evidence of a useful alloy search speed-up. In independent resamples of the spec-tiered seeds the target can be below the starting mean, producing a *zero-cost* crossing. The supplied `_ratio` floors the method's cost at **$250**, giving a finite **397×** upper percentile ($99,250/$250); this is a convention, not observed spend. A 20,000-replicate bootstrap with a literal zero-cost crossing has an infinite upper percentile for that comparison. The baseline first crossing can fluctuate near $100k by construction, and target, crossing and method rankings are resampled jointly; such percentile intervals are discrete, nonlinear and not general-purpose uncertainty bands. Bootstrap seeds as paired blocks when the same seed data matter, and report matched point/interval with the same matched sample. Comparisons remain descriptive because policy searches, calibration and target were selected in this specific benchmark.

### Paired pKGFN tests — **executed**

The table lists production-setting pKGFN minus each named policy; W/T/L denotes wins/ties/losses over the **same seven seeds**. These reproduce all 24 Wilcoxon rows in `tables.md` and README (exact two-sided p values rather than rounded).

{wtable}

For the most relevant *paired fidelity* comparison, high versus light scores at $100k were **58.856±12.208% versus 64.192±4.671%** on the seven common seeds; mean paired difference **−5.337 points**, Wilcoxon **p=0.6875**, paired seed-bootstrap 90% interval **−23.784 to +11.721 points** (50,000 draws, random seed 20261027). They have 3/0/4 high-setting wins/ties/losses. At $25k the difference is −11.323 (p=0.578125), at $50k −2.099 (p=0.9375). On those same seven seeds full BOFN ends at **87.367±3.878%**, production pKGFN minus full BOFN **−28.511** points (1/0/6, p=0.046875). The 7-seed high result is vulnerable to seed selection and cannot establish equivalence or that lighter Monte Carlo settings played no role. Even the seven high-setting `wall_s` total range **32.66–83.51 min** versus matched light mean **11.23 min**; timings are observational and may reflect a different machine/load.

**Multiplicity:** these are exploratory comparisons over eight rivals × three budgets (24 tests): Holm adjustment makes the minimum adjusted p **0.375**, and production-pKGFN-versus-full BOFN at $100k adjusts to **0.84375**. Thus nominal p=0.047 must not be advertised as a multiplicity-robust finding. In contrast, full BOFN versus `tiered_ni_ref` at $100k uses all 20 matched seeds: **+48.979 points, 17/0/3, p=0.000322**; even conservatively correcting the 9-policy, 3-budget all-pairs family (108 tests) by Holm gives **0.0274**. The choice of comparison family is discretionary; both families and exact tests are in `outputs/independent_paired.csv` and `outputs/all_policy_pairs.csv`.

## 2. Supplied curve/cost/seed logic — **executed and inspected**

I generated an adversarial five-row example at spends $500, $1,000 (two rows), $100,000, and $100,010. `analyze.curve` gives **NaN at $0 and $250** (before first row), **0.2 at $500/$750**, **0.3 at exactly $1,000** (the last duplicate-spend row and running maximum), **0.3 at $99,750**, and **0.4 at exactly $100,000**; it excludes the $100,010 row. `cost_to` returns **$500, $1,000, $100,000, ∞** for targets 0.2, 0.3, 0.4, 0.5. This is right-continuous and grid-rounded upward, not a linear interpolation. Every actual JSON starts with an `init` row at $0, so actual pre-first-row behavior is never exercised; all 179 scored-run initial rows are present. I found **no decreases** in stored `best_validated`, and the independent curves and supplied curves match **exactly (max absolute difference zero)**. There is a deliberate `curve` exception: `rec_printed` is *not* made monotonic (it describes the current model's recommendation, not best demonstrated value). A first-crossing helper uses a 1e-12 tolerance for mean curves, whereas `cost_to` uses a strict comparison; no printed benchmark change arose from that difference.

`matched_idx` and `paired` sort/intersect explicit seed IDs, not positional indices. I tested deliberately shuffled IDs `[7,0,2]` against `[2,3,7]` and obtained pairs `(2,2),(7,7)`, then checked all actual run keys are unique. `paired` calls the two-sided signed-rank test and includes zero differences via SciPy's default zero handling. The speed-up's matched bootstrap does align IDs, but switches to their intersection for both curves; it does **not** preserve the 20-seed baseline target from the mixed-sample estimate.

## 3. Target sensitivity, inference and wording — **executed plus policy interpretation inspected**

For fixed target levels of the (1,1) optimum, first crossing of each **mean** curve is:

{fixed_table}

At fixed **55%**, production pKGFN reaches its target only at the $100k endpoint, versus $76,250 for full BOFN and $65,750 for reuse; at **60% and 65% its seven-seed mean never reaches the target**, while full BOFN needs $85,750 at both 60% and 65% and reuse needs $71,000 and $81,000, respectively. The low 24.23% reference target and the negative spec-tiered target strongly affect the apparent “speed-up.” No single speed-up factor is stable across scientifically plausible outcome levels. A first-crossing of a mean is **not** the median first-crossing of seeds (e.g. production pKGFN mean never reaches 60%, but its per-seed median 60% cost is $100k).

A **standard error** is not a 90% confidence interval. The supplied `±` is further understated by ddof=0; selection of only completed production seeds is not included in any interval. Percentile bootstraps are non-smooth near no crossing, zero spend and the terminal grid point. Interpret their quantiles conditionally on the stored seeds and the chosen simulator, not as proof that an implementation generalizes to new physical alloys. README's “aluminum headline does not carry over” is fair *for this stored implementation*, but “the reason is clear: the immersion gate” overstates identification: the no-gate comparison changes policy choices as well as the objective; only random holds designs fixed. The causal assertion that BOFN “learns [the gate] from intermediate measurements” is a plausible mechanism, **not measured directly by the JSONs**. Synthetic hand-chosen functions, noiseless observations, an uncalibrated deposit-quality index (not literal relative density), an optimistic known-reference-coupon screen, and no physical data make external/physical performance an open question. The optimum-fails-CALPHAD restriction is verifiable from `truth_tables.json`; the one-hour screen does **not** exclude the optimum, as the README itself notes. Thus I would not attribute all copper-vs-aluminum differences to an immersion screen or claim the best method is partial stage selection.

**Most defensible single preliminary-results sentence:** “In a synthetic, deliberately biased copper directed-energy-deposition benchmark, full-pipeline function-network Bayesian optimization demonstrated alloys averaging 73% of the modeled optimum after $100,000 of new purchases (20 seeds), versus 24% for corrosion-informed tiered screening (20 seeds) and 14% for standard Bayesian optimization (20 seeds); a seven-seed production-setting partial-stage knowledge-gradient implementation averaged 59% and did not establish an advantage over full-pipeline search.”

As a supplemental check, I transcribed the no-gate truth into NumPy and independently optimized it by three SciPy differential-evolution starts: **0.74505147885**. The 80 no-gate files reproduce README's mean $100k percentages (BOFN reuse 87.555%, full 84.774%, standard BO 65.156%, random 50.867%); the 26 random-design records per seed are identical between gate/no-gate. Other policies choose different designs, so this is a *compound ablation*, not an identified share of their performance loss.

{ng_table}

## 4. The uploaded proposal PNG — **actual raster read and figure code inspected**

`bofn_cu_best_vs_spend_k1_partial_vs_full.png` is an 11×6.2-inch, 300-dpi raster. I read positions against its graph coordinates and checked its curves against the independent JSON reconstructions; raster-level values are approximate (typically within ~0.6 point). The pKGFN red solid curve is **light-settings, 20 seeds**, not the production-setting seven-seed curve. The plot has five policies, with no plotted corrosion-aware baseline:

| Trace in uploaded PNG | Visual ~25k / 50k / 100k (% optimum) | Exact independent curve at 25k / 50k / 100k |
|---|---|---|
| Solid red “Partial: pKGFN” (light) | ~12 / 27 / 59 | 11.951 / 27.440 / 58.552 |
| Dashed red “Partial: BOFN + reuse” | ~14 / 32 / 72 | 14.353 / 32.274 / 72.033 |
| Dark “Full: BOFN” | ~15 / 39 / 73 | 14.977 / 38.774 / 73.212 |
| Light gray “Full: standard BO” | ~6 / 7 / 14 | 5.680 / 7.180 / 14.421 |
| Gray “Tiered screening” (spec) | ~−9 / −9 / −8 | −9.427 / −9.106 / −7.691 |

The start level for the 20-seed traces is **−9.610%** (axis label −10); the current production-figure variant starts at **−13.831%** on its seven pKGFN seeds (axis label −14). **There is no called-out spend or horizontal target level in this uploaded figure**: plot code disables the callout when the baseline is the uninformative negative-valued spec-tiered rule. Its *unplotted* corresponding pair would be **$5,750 and −7.691%** for light pKGFN versus **$99,250 and −7.691%** for spec tiered; for the corrosion-informed comparison it would be **$47,250 and +24.233%** for light pKGFN versus **$100,000 and +24.233%** for the reference baseline. Do not invent a callout from a figure that has none.

At 3.1 inches wide, the plotting code scales its nominal 24-pt source labels to ~6.8 pt, and plot data use ~53% of the source width (~1.64 inches at proposal size). Its x axis labels only 0 and 100, its y axis only −10 and 73, with **no zero line**. Light gray standard BO and near-flat tiered traces can disappear into the background. Red and spec-tiered **±one (population-based) SE** ribbons appear, while full BOFN and standard BO lack ribbons: do not read missing shading as greater precision. Two red traces suggest “partial” has a single benefit, though BOFN reuse tests every purchased Mines build downstream, and pKGFN is the actual stage selector. Add `light n=20` (or replace with the current n=7 curve), state bands/seed counts, include a visible 0% line and intermediate spend ticks, and distinguish reuse from stage selection. If claiming the 3× result, use the corrosion-informed reference baseline, not this old plot; label its exact comparator and target.

## 5. Purchases and run pathology — **executed; mechanisms inspected**

I reconciled **all 179 main+light run ledgers exactly** against eight stage counts, the $1,600-per-paid-scale-up capacity of four sessions per lot, the five pre-existing seed lots, and final stored spend. Every new build count equals new Mines purchases; each stage completion count S/T/K/I equals its populated build field. This independently checks `tables.md`'s purchase table below (values are **my unrounded means per run**, excluding five shared seed builds and lots; the supplied table rounds to one decimal). `C` denotes CALPHAD, `A` atomization, `U` Utah small deposition, `M` Mines large deposition, `S` microscopy, `T` tensile, `K` one-hour screen, `I` immersion.

{purchase_table}

- **What was bought.** BOFN full, standard BO and random run ~21 complete eight-stage chains (standard BO saves 1.3 CALPHAD calls on previously known compositions). BOFN reuse runs **12.55 new lots and 29.30 fully tested Mines sessions**. Spec tiered performs **224 new CALPHAD screens**, ~22.8 atomizations, 44.25 Utah visits, 24.85 Mines sessions, only **4.0 immersions**. Ni-floor tiered performs 31.0 lots, 67.4 Utah visits, 18.35 Mines sessions and **2.55 immersions**. Reference-gated Ni tiered performs 26.5 lots, 53.65 Utah visits, 21.15 Mines sessions, **9.9 immersions** with **2.65 hard passes** on average. Production pKGFN does only **4.57 CALPHAD calls**, but **37.86 Mines sessions on 17.57 new lots**, only **5.86 tensile** and **11.29 immersion** purchases, and **5.29 completely validated new builds**; buying immersion and tensile on different builds does not validate both. The five shared fully tested seed builds are excluded throughout.
- **Repeated purchases:** no duplicate exact stored builds or Utah sessions in any main/light run; no duplicate lots except *standard BO seed 19*, which atomizes exactly the same serialized `(composition, atomizer setting)=(0,0,1,0,0)` a second time (lot indices 14 and 20). These are checks of stored/rounded identities, not proof of equality at unlimited floating precision. There are many **intentional repeat Mines sessions** on distinct settings of the same lot. The cost ledger does not show free/reused identical purchases by accident.
- **Budget and stalls:** all headline runs end between **$97,790 and $100,000**; none ends far below $100k. Spec tiered has **15/20 runs with zero improvement from the initial validated best**, Ni-floor tiered **11/20**, reference-gated tiered **3/20**, random **4/20**. High-setting pKGFN can spend up to **$77,900** in a contiguous no-improvement run (seed 2); the longest contiguous run of new actions costing ≤$500 each without improving the validated best is **10 actions/$3,510** (seed 5). Light pKGFN has up to **20 such actions/$5,570** (seed 15) or **16 actions/$7,900** (seed 14). At $99,750 the production pKGFN mean is **46.214%**, versus **58.856%** at $100k, because seeds **3, 5, 7** finalize gains of ~20.77, 48.01 and 19.70 points during that last grid interval. The end figure and $100k medians therefore depend strongly on near-budget validation.
- **Unfinished builds:** production pKGFN deposits ~37.86 new builds but validates ~5.29; ~26.00 new builds have **neither tensile nor immersion**, ~32.00 have no tensile and ~26.57 have no immersion. Light pKGFN: 40.50 deposited, 6.15 validated, 26.60 with neither test. Full BOFN validates all 21. For an upper bound using the synthetic truth on *all* stored deposited designs (including five seed designs), my independently transcribed oracle scores production pKGFN **70.817%** versus **58.856% demonstrated**, light **68.496%** versus **58.552% demonstrated**. This is an unavailable oracle, not prospective achievable policy performance; serialized compositions rounded to four decimals make the re-evaluated oracle approximate.

**Flat knowledge-gradient (KG) diagnostic, production pKGFN:**

{fallback_table}

Production pKGFN has **267/689=38.75%** fallback-marked purchases (README's “39%” and `tables.md` pooled 26% across high and light reproduce), spending ~**36.08%** of its new budget on flagged purchases. On seeds 1, 2 and 7 the fallback takes **70.1%, 67.0%, 88.6%** of their actions (seed 4 takes 26.7%; other seeds ~1.2%, 3.3%, 4.9%). In the before-state J≤0, **255/383=66.6%** of purchases are fallback, compared with **12/306=3.9%** after first J>0; light has **380/907=41.9%** before and **38/1,037=3.7%** after. Source code marks fallback when estimated KG per dollar is below **1e−10** or no eligible KG candidate remains, and buys a one-step action using a path-priced EI heuristic or an optimistic posterior sample. The flag is evidence of *this discretized numerical acquisition being near-flat*, **not** proof mathematical knowledge gradient is exactly zero, nor proof the fallback buys an optimal path. The flag does not indicate which of its two fallback subbranches was used; action labels cannot identify that. It is consistent with delayed rewards and an unfinished-validation backlog, but this is mechanistic interpretation, not a controlled intervention.

## 6. Sensitivity cells and proposed new policies — **executed where files exist; remainder inspected only**

At (0,0), normalized by `truth_tables.json`'s optimum **0.381561510051002**, I independently find:

{sens_table}

These reproduce the rounded (0,0) table: light pKGFN −6±4, approximately 0±4, 9±7 (the $25k **sample** SE is ~5, not 4); full BOFN 12,19,92; reuse 12,68,95. **n=1 means no between-seed SE for either BOFN row and no valid method comparison.** No stored JSONs exist for sensitivity cells (1,0) or (0,1), and (0,0) has no production pKGFN, tiered, standard BO, random, screen/audit, mine-only or funnel result. `tables.md`'s populated sensitivity entries agree at its reporting precision; blank cells really mean absent runs, not zero performance. README's phrases “both screens right” and “robust to screen bias” cannot be inferred from these sparse cells.

I enumerated every kind in `results/` and `quality/results-light/`: **no `pkgfn_screen`, `pkgfn_audit`, `mines_only` or `funnel` files**. Consequently the screen-audit and fixed-funnel result tables are empty and the two-step DED split table is not rendered at all in `tables.md`; I **inspected only** their `analyze.py` implementations and README policy descriptions, and did not manufacture outcome values. In particular, the audit method's final-gate truth check and near-cutoff oversampling cannot be validated as an empirical estimator without audit logs. The code's current implementation offers *questions to test*, not evidence that a Utah step, a trusting screen, auditing or fixed top-k funnel helps. The same caveat applies to the unfinished S=256 fidelity check.

## Verification record, artifacts, and limits

| Check | Status | Evidence |
|---|---|---|
| Read settings, exact optima, inventories; reconstruct all 179 main/light curves with NumPy before opening supplied aggregator | **Executed** | `outputs/independent_core.py`, `outputs/independent_summary.csv`; max supplied-versus-independent curve error = **0** |
| Headline means, sample SE, medians, paired Wilcoxon, both baseline speed-ups and both bootstrap schemes | **Executed** | `outputs/all_SE_comparisons.csv`, `independent_paired.csv`, `independent_speedups.csv`, `replicated_bootstrap.csv`; exact 2,000-replicate bootstrap replication matches supplied |
| Regenerate supplied `tables.md` with pure NumPy analysis functions while avoiding torch import/new truth optimization | **Executed** | `outputs/supplied_tables_regenerated.md`; **zero differing lines** versus uploaded `tables.md` |
| Synthetic step-function edge cases; shuffled seed IDs; original cost-and-capacity reconciliation; build statuses and actions | **Executed** | Edge-case test outputs; `outputs/inventory_cost_ledger.csv` (179/179 exact), `action_audit_per_seed.csv` |
| Target sensitivity, Holm corrections, matched high-v-light comparisons and paired-bootstrap interval | **Executed** | `fixed_target_costs.csv`, `all_policy_pairs.csv`, `independent_paired.csv`; bootstrap seed 20261027 |
| Actual original PNG and plot code; approximate trace read-offs | **Executed raster/inspected code** | `outputs/figure_raster_readings.csv`; displayed pixel checks above |
| No-gate optimum and four-policy JSON summary; random same-design check | **Executed** | `outputs/nogate_summary.csv`; SciPy/NumPy truth transcription; no aluminum JSONs supplied, so prior aluminum comparison **not re-executed** |
| Metallurgical plausibility, BoTorch/GP implementations, F2/F4 correctness, physical density calibration, new policies' behavior | **Inspected only / not established** | Source descriptions; no new training, experimental measurements, or absent run files |

I treated the seven completed high-fidelity pKGFN seeds as a **nonrandom convenience subset**; intervals conditional on them do not account for incomplete uploads. I chose sample SD for a conventional between-seed SE (instead of retaining the supplied population SD), both independent and matched seed bootstraps to expose different estimands, and a 108-test all-policy family only as a conservative multiplicity sensitivity check. No web source was needed; every factual input here is a supplied file or an executed reconstruction. Reproducibility and detailed decimal-level audits are in `outputs/`; original inputs were not modified.

**Discretionary analytical decisions**
- I used the exact uploaded optima rather than re-optimizing the gated truth: the requested metric depends on the benchmark's stipulated normalization; the separate no-gate optimum required an independent reconstruction because its landscape JSON was absent.
- I report sample-SD standard errors rather than the supplied population-SD convention, since seeds represent replicate runs and n=1 cannot estimate a between-seed error; I preserve the supplied numbers side by side for auditability.
- I keep `∞` for censored threshold costs and call zero-cost bootstrap crossings undefined/infinite instead of interpreting the supplied $250 floor as experimental spend.
- I present matched and independent resampling separately and explicitly recompute the matched-sample point estimate, because matching changes the target when one policy has 7 rather than 20 seeds.
- I use fixed 55/60/65% targets alongside baseline-endpoint targets to test sensitivity without substituting a favorable single threshold, and apply Holm corrections as a conservative illustration rather than post hoc confirmatory inference.
- I characterize missing runs as unavailable, not poor results, and describe biological/physical explanations as hypotheses because the task supplies a hand-built noiseless simulator, not measured validation.
'''
(out/'report.md').write_text(report)
print('report file',out/'report.md','chars',len(report),'lines',len(report.splitlines()))
print('listed audit files',[(p.name,p.stat().st_size) for p in sorted(out.iterdir())])


report file /workspace/outputs/report.md chars 38880 lines 238
listed audit files [('SE_rounding_changes.csv', 1302), ('action_audit_per_seed.csv', 22449), ('all_SE_comparisons.csv', 7131), ('all_policy_pairs.csv', 8790), ('figure_raster_readings.csv', 685), ('fixed_target_costs.csv', 387), ('independent_core.py', 2210), ('independent_paired.csv', 1601), ('independent_speedups.csv', 2619), ('independent_summary.csv', 2208), ('inventory_cost_ledger.csv', 11449), ('nogate_summary.csv', 892), ('quality_checks.json', 303), ('replicated_bootstrap.csv', 1938), ('report.md', 39326), ('supplied_tables_regenerated.md', 11039)]


In [23]:
# Proofread and correct two drafting errors before delivery.
report=report.replace('seven production pKGFN and eight other 20-seed headline policies','seven production pKGFN and seven other 20-seed headline policies')
report=report.replace('for the corrosion-informed comparison it would be **$47,250 and +24.233%** for light pKGFN','for the corrosion-informed comparison it would be **$46,750 and +24.233%** for light pKGFN')
report=report.replace('biological/physical explanations','physical explanations')
# Display exact rounded-SE changes in the matched fidelity table too.
needle='They have 3/0/4 high-setting wins/ties/losses.'
report=report.replace(needle,'The matched-light README row should have sample-SE displays **±11, ±12, ±5**, rather than **±10, ±12, ±4** at $25k/$50k/$100k; its exact means are **14.664%, 26.671%, 64.192%**. '+needle)
(out/'report.md').write_text(report)
# Reproducibility: save all successfully executed analysis cells, plus report generation.
# The final proofreading cell is appended to the script separately without export recursion.
hist=IPython.get_ipython().history_manager.input_hist_raw
prior=hist[1:-1]
script='# Reproduce from /workspace with numpy, scipy, pandas, pillow. No torch imported.\n# All paths refer to the uploaded workspace; inputs remain unchanged.\n\n'+'\n\n'.join(f'# === Executed notebook cell {i} ===\n'+code for i,code in enumerate(prior))
script += '\n\n# Final report proofreading corrections\n'+hist[-1].split('# Reproducibility:')[0]+'\n'
(out/'audit_reproduce.py').write_text(script)
import nbformat
nb=nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell(code) for code in prior])
nbformat.write(nb,str(out/'audit_notebook.ipynb'))
print('Saved',len(prior),'executed code cells into audit_reproduce.py and audit_notebook.ipynb')
print('Final report length:',len(report),'characters')
# Sanity checks on all output claims that are easy to encode.
assert len(list(Path('results').rglob('*.json')))==149
assert len(runs[1,1,'pkgfn'])==7
assert all(len(runs[1,1,p])==20 for p in POL if p!='pkgfn')
assert (out/'report.md').read_text()==report
assert '$46,750 and +24.233%' in report
print('Final saved-report equality and inventory assertions passed')


Saved 24 executed code cells into audit_reproduce.py and audit_notebook.ipynb
Final report length: 39047 characters
Final saved-report equality and inventory assertions passed


In [24]:
import subprocess, sys
# Exclude kernel-initialization magics and notebook-only source introspection.
analysis_cells=prior[2:]
script='# Reproduce from /workspace with numpy, scipy, pandas, pillow and IPython. No torch imported.\n\n'+'\n\n'.join(f'# === Analysis cell {i} ===\n'+code for i,code in enumerate(analysis_cells))
script=script.replace("Path(out/'independent_core.py').write_text(IPython.get_ipython().history_manager.input_hist_raw[-1].split('# Save the independent')[0])", "# independent_core.py was saved before opening analyze.py in the original notebook.")
proofread=hist[-1].split('# Reproducibility:')[0] if hist[-1].startswith('# Proofread') else "report=report.replace('seven production pKGFN and eight other 20-seed headline policies','seven production pKGFN and seven other 20-seed headline policies')\nreport=report.replace('for the corrosion-informed comparison it would be **$47,250 and +24.233%** for light pKGFN','for the corrosion-informed comparison it would be **$46,750 and +24.233%** for light pKGFN')\nreport=report.replace('biological/physical explanations','physical explanations')\nneedle='They have 3/0/4 high-setting wins/ties/losses.'\nreport=report.replace(needle,'The matched-light README row should have sample-SE displays **±11, ±12, ±5**, rather than **±10, ±12, ±4** at $25k/$50k/$100k; its exact means are **14.664%, 26.671%, 64.192%**. '+needle)\n(out/'report.md').write_text(report)"
script += '\n\n# Final report proofreading corrections\n'+proofread+'\n'
(out/'audit_reproduce.py').write_text(script)
nb=nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell(code) for code in analysis_cells]+[nbformat.v4.new_code_cell(proofread)])
nbformat.write(nb,str(out/'audit_notebook.ipynb'))
check=subprocess.run([sys.executable,str(out/'audit_reproduce.py')],cwd='/workspace',stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True,timeout=600)
(out/'reproduction_stdout.txt').write_text(check.stdout)
(out/'reproduction_stderr.txt').write_text(check.stderr)
print('Fresh-process reproduction exit:',check.returncode)
print('stderr tail:',check.stderr[-2500:])
print('stdout tail:',check.stdout[-900:])
assert check.returncode==0
assert (out/'report.md').read_text()==report
print('Fresh-process numerical audit completed, delivered report preserved exactly.')

Fresh-process reproduction exit: 0
stderr tail: 
stdout tail: | 7.180 ± 3.299 | 7 ± 3 | 14.421 ± 6.256 | 14 ± 6 |
| Random | 20 | -0.612 ± 4.532 | -1 ± 4 | 6.904 ± 4.977 | 7 ± 5 | 22.293 ± 5.024 | 22 ± 5 |
report file /workspace/outputs/report.md chars 38880 lines 238
listed audit files [('SE_rounding_changes.csv', 1302), ('action_audit_per_seed.csv', 22449), ('all_SE_comparisons.csv', 7131), ('all_policy_pairs.csv', 8790), ('audit_notebook.ipynb', 81413), ('audit_reproduce.py', 72498), ('figure_raster_readings.csv', 685), ('fixed_target_costs.csv', 387), ('independent_core.py', 2210), ('independent_paired.csv', 1601), ('independent_speedups.csv', 2619), ('independent_summary.csv', 2208), ('inventory_cost_ledger.csv', 11449), ('nogate_summary.csv', 892), ('quality_checks.json', 303), ('replicated_bootstrap.csv', 1938), ('report.md', 39326), ('reproduction_stderr.txt', 425), ('reproduction_stdout.txt', 0), ('supplied_tables_regenerated.md', 11039)]

Fresh-process numerical audit complet

In [25]:
# Mark successful clean-process reproduction in the delivered document and its reproduction script.
verification='\n**Reproducibility check:** I also executed `outputs/audit_reproduce.py` in a fresh Python process: exit code 0, no stderr; the generated final report matched exactly. The replayable notebook is `outputs/audit_notebook.ipynb`.\n'
report=report.replace('\n**Discretionary analytical decisions**',verification+'\n**Discretionary analytical decisions**')
(out/'report.md').write_text(report)
with (out/'audit_reproduce.py').open('a') as f:
 f.write('\nverification='+repr(verification)+'\n')
 f.write("report=report.replace('\\n**Discretionary analytical decisions**',verification+'\\n**Discretionary analytical decisions**')\n(out/'report.md').write_text(report)\n")
print('Final report:',(out/'report.md').stat().st_size,'bytes')
print('Fresh-process reproduction logs:',check.returncode,repr(check.stderr))


Final report: 39726 bytes
Fresh-process reproduction logs: 0 ''
